# Rectangular RCC Hopper Bunker – Design Software (Starter Version)

**Platform:** Google Colab · **Design framework:** Indian Standards (IS)

This notebook starts with an interactive input form and a transparent geometry/load calculation engine. It is deliberately a **Stage 1 engineering tool**, not a certified structural design program.

## Important engineering scope
- IS 4995 (Part 1):1974 addresses general requirements and assessment of loads for reinforced-concrete bins storing granular/powdery materials; Part 2:1974 addresses design criteria. Confirm current reaffirmation/amendments and obtain the applicable BIS text before design.
- IS 456:2000 (with applicable amendments) is the general RCC design code.
- A rectangular hopper bunker needs project-specific assessment of filling, discharge, eccentric flow, wall friction, hopper pressures, supports, seismic/wind effects and load combinations.
- The notebook does **not** invent IS 4995 pressure coefficients or claim code compliance. It calculates geometry, capacity and self-weight, and provides clearly labelled *screening* pressure estimates only. Final bin/hopper pressures must be established from the applicable standard and project material data by a qualified structural engineer.

### Geometry convention
The hopper is modelled as a rectangular pyramidal frustum:
- top dimensions = internal bunker length × internal bunker width
- bottom outlet dimensions = outlet length × outlet width
- hopper height = vertical distance from hopper top to outlet plane
- vertical-wall height is measured above the hopper top

In [ ]:
# Install once in Google Colab
!pip -q install ipywidgets plotly

## 1. Imports and helper functions

In [ ]:
import math
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import plotly.graph_objects as go

def frustum_volume_rectangular(Lt, Wt, Lb, Wb, H):
    """Volume of a rectangular pyramidal frustum with geometrically similar plan sections.
    Formula: V = H/3 * (A1 + A2 + sqrt(A1*A2)).
    For non-similar top/bottom rectangles this is an approximation; use CAD/integration
    for exact geometry where side slopes differ in the two directions.
    """
    A1, A2 = Lt*Wt, Lb*Wb
    return H/3.0 * (A1 + A2 + math.sqrt(A1*A2))

def sloping_panel_area(top_a, bottom_a, height):
    """Area of one trapezoidal sloping panel, assuming straight side profile."""
    return 0.5 * (top_a + bottom_a) * math.sqrt(height**2 + ((top_a-bottom_a)/2.0)**2)

def bunker_geometry(inp):
    Li = inp["L_outer"] - 2*inp["t_wall"]
    Wi = inp["W_outer"] - 2*inp["t_wall"]
    if Li <= 0 or Wi <= 0:
        raise ValueError("Wall thickness leaves no positive internal plan dimensions.")
    if inp["L_outlet"] <= 0 or inp["W_outlet"] <= 0:
        raise ValueError("Outlet dimensions must be positive.")
    if inp["L_outlet"] >= Li or inp["W_outlet"] >= Wi:
        raise ValueError("Outlet dimensions must be smaller than internal hopper-top dimensions.")
    if inp["H_wall"] <= 0 or inp["H_hopper"] <= 0:
        raise ValueError("Wall and hopper heights must be positive.")

    V_vertical = Li * Wi * inp["H_wall"]
    V_hopper = frustum_volume_rectangular(Li, Wi, inp["L_outlet"], inp["W_outlet"], inp["H_hopper"])
    V_total = V_vertical + V_hopper

    # Four hopper panels (two long-side panels and two short-side panels)
    A_hopper = (
        2 * sloping_panel_area(Wi, inp["W_outlet"], inp["H_hopper"])
        + 2 * sloping_panel_area(Li, inp["L_outlet"], inp["H_hopper"])
    )
    A_wall = 2*(Li+Wi)*inp["H_wall"]
    A_base = inp["L_outlet"]*inp["W_outlet"]

    return {
        "L_internal": Li, "W_internal": Wi,
        "A_internal": Li*Wi,
        "V_vertical": V_vertical, "V_hopper": V_hopper,
        "V_capacity": V_total, "A_vertical_walls": A_wall,
        "A_hopper_sloping": A_hopper, "A_outlet": A_base
    }

def calculate_stage1(inp):
    g = bunker_geometry(inp)
    gamma_c = inp["gamma_concrete"]
    gamma_m = inp["gamma_material"]

    # Approximate concrete volumes: vertical walls as four constant-thickness plates;
    # hopper slab as sloping surface area × thickness. Corner overlaps are approximate.
    V_walls_conc = g["A_vertical_walls"] * inp["t_wall"]
    V_hopper_conc = g["A_hopper_sloping"] * inp["t_hopper"]
    V_outlet_ring = 0.0  # outlet ring/beam not included until designed separately
    W_walls = V_walls_conc * gamma_c
    W_hopper = V_hopper_conc * gamma_c

    W_material = g["V_capacity"] * gamma_m
    # Mean vertical pressure at the bottom/outlet footprint is a crude load/area indicator,
    # not a design pressure for hopper plates or supports.
    q_equiv = W_material / max(g["A_outlet"], 1e-9)

    # Screening-only triangular lateral pressure: user-entered K_screen.
    # Not an IS 4995 design pressure; ignores Janssen effects, flow pattern and discharge.
    p_screen_base = inp["K_screen"] * gamma_m * inp["H_wall"]
    M_screen_per_m = p_screen_base * inp["H_wall"]**2 / 6.0

    return {
        **g,
        "V_walls_concrete": V_walls_conc,
        "V_hopper_concrete": V_hopper_conc,
        "W_walls": W_walls,
        "W_hopper": W_hopper,
        "W_material": W_material,
        "W_concrete_total": W_walls + W_hopper,
        "W_total_simple": W_material + W_walls + W_hopper,
        "q_equiv_outlet": q_equiv,
        "p_screen_base": p_screen_base,
        "M_screen_per_m": M_screen_per_m
    }

## 2. Interactive input form

Enter dimensions in metres, unit weights in kN/m³, and angles in degrees. The material properties are user-defined because the software is intended to accept different stored materials.

**Note:** internal friction angle and wall-friction angle are recorded for the later IS 4995 load module; they are not used to fabricate a design pressure in this starter version.

In [ ]:
# Geometry
L_outer = widgets.FloatText(value=6.0, description="Outer length (m)", style={"description_width":"initial"})
W_outer = widgets.FloatText(value=4.0, description="Outer width (m)", style={"description_width":"initial"})
H_wall = widgets.FloatText(value=5.0, description="Vertical wall H (m)", style={"description_width":"initial"})
t_wall = widgets.FloatText(value=0.25, description="Wall thickness (m)", style={"description_width":"initial"})
H_hopper = widgets.FloatText(value=2.0, description="Hopper height (m)", style={"description_width":"initial"})
L_outlet = widgets.FloatText(value=1.0, description="Outlet length (m)", style={"description_width":"initial"})
W_outlet = widgets.FloatText(value=1.0, description="Outlet width (m)", style={"description_width":"initial"})
t_hopper = widgets.FloatText(value=0.20, description="Hopper slab thickness (m)", style={"description_width":"initial"})

# Material properties
material_name = widgets.Text(value="User-defined bulk material", description="Material", style={"description_width":"initial"})
gamma_material = widgets.FloatText(value=8.0, description="Bulk unit wt (kN/m³)", style={"description_width":"initial"})
phi_internal = widgets.FloatText(value=30.0, description="Internal friction φ (°)", style={"description_width":"initial"})
delta_wall = widgets.FloatText(value=20.0, description="Wall friction δ (°)", style={"description_width":"initial"})
gamma_concrete = widgets.FloatText(value=25.0, description="Concrete unit wt (kN/m³)", style={"description_width":"initial"})

# RCC properties (captured now for later design modules)
fck = widgets.Dropdown(options=[("M20",20),("M25",25),("M30",30),("M35",35),("M40",40),("M45",45),("M50",50)], value=25, description="Concrete grade", style={"description_width":"initial"})
fy = widgets.Dropdown(options=[("Fe415",415),("Fe500",500),("Fe550",550)], value=500, description="Steel grade", style={"description_width":"initial"})
cover = widgets.FloatText(value=40.0, description="Nominal cover (mm)", style={"description_width":"initial"})

# Optional screening parameter only; not a code-derived coefficient
K_screen = widgets.FloatText(value=0.50, description="K screening only", style={"description_width":"initial"})

form_items = [
    widgets.HTML("<h3>Geometry</h3>"),
    L_outer, W_outer, H_wall, t_wall, H_hopper, L_outlet, W_outlet, t_hopper,
    widgets.HTML("<h3>Stored material and concrete</h3>"),
    material_name, gamma_material, phi_internal, delta_wall, gamma_concrete,
    widgets.HTML("<h3>RCC properties (for later stages)</h3>"),
    fck, fy, cover,
    widgets.HTML("<h3>Optional preliminary screening input</h3>"),
    K_screen
]
form = widgets.VBox(form_items)
run_button = widgets.Button(description="Calculate geometry and loads", button_style="success", icon="calculator")
out = widgets.Output()

def get_inputs():
    return {
        "L_outer": L_outer.value, "W_outer": W_outer.value,
        "H_wall": H_wall.value, "t_wall": t_wall.value,
        "H_hopper": H_hopper.value, "L_outlet": L_outlet.value,
        "W_outlet": W_outlet.value, "t_hopper": t_hopper.value,
        "gamma_material": gamma_material.value,
        "phi_internal": phi_internal.value, "delta_wall": delta_wall.value,
        "gamma_concrete": gamma_concrete.value,
        "fck": fck.value, "fy": fy.value, "cover_mm": cover.value,
        "K_screen": K_screen.value, "material_name": material_name.value
    }

def on_run_clicked(_):
    with out:
        clear_output()
        try:
            inp = get_inputs()
            if any(v <= 0 for k,v in inp.items() if k not in ["phi_internal","delta_wall","K_screen"] and isinstance(v,(int,float))):
                raise ValueError("All dimensions and unit weights must be positive.")
            if not (0 <= inp["phi_internal"] < 60 and 0 <= inp["delta_wall"] < 60):
                raise ValueError("Check friction angles; enter plausible values in degrees.")
            if inp["K_screen"] < 0:
                raise ValueError("Screening coefficient cannot be negative.")

            r = calculate_stage1(inp)
            rows = [
                ("Internal length", r["L_internal"], "m"),
                ("Internal width", r["W_internal"], "m"),
                ("Vertical storage volume", r["V_vertical"], "m³"),
                ("Hopper storage volume", r["V_hopper"], "m³"),
                ("Total geometric capacity", r["V_capacity"], "m³"),
                ("Approx. stored material weight", r["W_material"], "kN"),
                ("Approx. vertical wall concrete weight", r["W_walls"], "kN"),
                ("Approx. hopper concrete weight", r["W_hopper"], "kN"),
                ("Simple total weight (material + walls + hopper)", r["W_total_simple"], "kN"),
                ("Equivalent material load / outlet area (indicator only)", r["q_equiv_outlet"], "kN/m²"),
                ("Screening lateral pressure at bottom of vertical wall", r["p_screen_base"], "kN/m²"),
                ("Screening cantilever moment per metre strip", r["M_screen_per_m"], "kN·m/m")
            ]
            df = pd.DataFrame(rows, columns=["Calculated item","Value","Unit"])
            display(df.style.format({"Value":"{:,.3f}"}))
            print("\nDesign inputs captured:")
            print(f"Material: {inp['material_name']} | fck = {inp['fck']} MPa | fy = {inp['fy']} MPa | cover = {inp['cover_mm']} mm")
            print("\nIMPORTANT: the last two screening values are NOT IS 4995 design actions.")
            print("Do not use them for reinforcement design. They omit bin pressure theory, filling/discharge cases, eccentricity, hopper flow and load combinations.")
        except Exception as e:
            print("INPUT / CALCULATION ERROR:", e)

run_button.on_click(on_run_clicked)
display(form, run_button, out)

Button(button_style='success', description='Calculate geometry and loads', icon='calculator', style=ButtonStyl…

Output()

In [ ]:
def get_all_inputs():

    return {
        "L_outer": L_outer.value,
        "W_outer": W_outer.value,
        "H_wall": H_wall.value,
        "t_wall": t_wall.value,

        "H_hopper": H_hopper.value,
        "L_outlet": L_outlet.value,
        "W_outlet": W_outlet.value,
        "t_hopper": t_hopper.value,

        "gamma_material": gamma_material.value,
        "phi_internal": phi_internal.value,
        "delta_wall": delta_wall.value,
        "gamma_concrete": gamma_concrete.value,

        "fck": fck.value,
        "fy": fy.value,
        "cover": cover.value,
        "K_screen": K_screen.value,

        "material_name": material_name.value
    }


# Check all entered inputs
inputs = get_all_inputs()

for key, value in inputs.items():
    print(f"{key}: {value}")

L_outer: 6.0
W_outer: 4.0
H_wall: 7.0
t_wall: 0.25
H_hopper: 2.0
L_outlet: 1.0
W_outlet: 1.0
t_hopper: 0.2
gamma_material: 8.0
phi_internal: 30.0
delta_wall: 20.0
gamma_concrete: 25.0
fck: 25
fy: 500
cover: 40.0
K_screen: 0.5
material_name: User-defined bulk material


## 3. Calculation basis and equations

### A. Internal plan dimensions
\[
L_i = L_o - 2t_w, \qquad W_i = W_o - 2t_w
\]

### B. Storage volume
Vertical portion:
\[
V_v = L_i W_i H_v
\]

Rectangular pyramidal frustum hopper (valid as an exact formula when top and outlet rectangles are geometrically similar; otherwise treated as a preliminary approximation):
\[
V_h = \frac{H_h}{3}\left(A_1+A_2+\sqrt{A_1A_2}\right)
\]

Total geometric capacity:
\[
V = V_v+V_h
\]

### C. Stored material weight
\[
W_m = V\gamma_m
\]

### D. Approximate concrete self-weight
Vertical wall plates:
\[
W_w \approx 2(L_i+W_i)H_v t_w\gamma_c
\]

Hopper plates:
\[
W_h \approx A_{hopper}t_h\gamma_c
\]

These are quantity estimates. They do not include corner overlap refinement, ring beams, outlet beams, columns, foundations, haunches, openings or other local thickening.

### E. Screening lateral pressure (NOT a design method)
The optional display uses:
\[
p_{screen}=K_{screen}\gamma_m H_v
\]
and a triangular-cantilever strip indicator:
\[
M_{screen}=\frac{p_{screen}H_v^2}{6}
\]
This is only a user-controlled sanity-check illustration. **It is not an IS 4995 pressure calculation** and must not be used to design bunker walls.

## 4. Interactive 3D geometry preview

This is a geometry preview only; reinforcement and structural supports are not yet modelled.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

!pip install -q plotly

In [ ]:

# ==========================================
# DYNAMIC 3D RECTANGULAR RCC HOPPER BUNKER
# ==========================================

from IPython.display import display, clear_output
import ipywidgets as widgets
import plotly.graph_objects as go

bunker_3d_out = widgets.Output()


def draw_dynamic_bunker(change=None):

    with bunker_3d_out:
        clear_output(wait=True)

        try:
            # Always read the latest original input values
            inp = get_inputs()

            L = inp["L_outer"]
            W = inp["W_outer"]
            t = inp["t_wall"]
            H = inp["H_wall"]
            Hh = inp["H_hopper"]
            Lo = inp["L_outlet"]
            Wo = inp["W_outlet"]

            # Calculate internal dimensions
            Li = L - 2*t
            Wi = W - 2*t

            if min(L, W, t, H, Hh, Lo, Wo) <= 0:
                print("All dimensions must be positive.")
                return

            if Li <= 0 or Wi <= 0:
                print("Wall thickness is too large for the bunker dimensions.")
                return

            if Lo >= Li or Wo >= Wi:
                print("Outlet dimensions must be smaller than internal dimensions.")
                return

            fig = go.Figure()

            # Function to draw a four-sided surface
            def add_quad(points, color, opacity=1.0):
                fig.add_trace(go.Mesh3d(
                    x=[p[0] for p in points],
                    y=[p[1] for p in points],
                    z=[p[2] for p in points],
                    i=[0, 0],
                    j=[1, 2],
                    k=[2, 3],
                    color=color,
                    opacity=opacity,
                    flatshading=True,
                    showscale=False
                ))

            # Outer and inner corners of vertical walls
            outer_base = [
                (-L/2, -W/2, Hh),
                ( L/2, -W/2, Hh),
                ( L/2,  W/2, Hh),
                (-L/2,  W/2, Hh)
            ]

            inner_base = [
                (-Li/2, -Wi/2, Hh),
                ( Li/2, -Wi/2, Hh),
                ( Li/2,  Wi/2, Hh),
                (-Li/2,  Wi/2, Hh)
            ]

            outer_top = [
                (x, y, Hh + H) for x, y, z in outer_base
            ]

            inner_top = [
                (x, y, Hh + H) for x, y, z in inner_base
            ]

            # Draw four outer wall faces, four inner faces and top rim
            for i in range(4):
                j = (i + 1) % 4

                add_quad(
                    [outer_base[i], outer_base[j],
                     outer_top[j], outer_top[i]],
                    "#6495ED", 0.85
                )

                add_quad(
                    [inner_base[j], inner_base[i],
                     inner_top[i], inner_top[j]],
                    "#6495ED", 0.65
                )

                add_quad(
                    [outer_top[i], outer_top[j],
                     inner_top[j], inner_top[i]],
                    "#4169A1", 0.95
                )

            # Hopper outlet corners
            bottom = [
                (-Lo/2, -Wo/2, 0),
                ( Lo/2, -Wo/2, 0),
                ( Lo/2,  Wo/2, 0),
                (-Lo/2,  Wo/2, 0)
            ]

            # Four sloping hopper panels
            for i in range(4):
                j = (i + 1) % 4

                add_quad(
                    [bottom[i], bottom[j],
                     inner_base[j], inner_base[i]],
                    "#E6A23C", 0.95
                )

            # Draw visible edges
            def draw_edges(points):
                fig.add_trace(go.Scatter3d(
                    x=[p[0] for p in points],
                    y=[p[1] for p in points],
                    z=[p[2] for p in points],
                    mode="lines",
                    line=dict(color="black", width=3),
                    showlegend=False
                ))

            draw_edges(outer_top + [outer_top[0]])
            draw_edges(inner_top + [inner_top[0]])
            draw_edges(bottom + [bottom[0]])

            for i in range(4):
                draw_edges([outer_base[i], outer_top[i]])
                draw_edges([inner_base[i], inner_top[i]])
                draw_edges([bottom[i], inner_base[i]])

            fig.update_layout(
                title=(
                    "DYNAMIC RECTANGULAR RCC HOPPER BUNKER"
                    f"<br><sup>Outer L={L} m | Outer W={W} m | "
                    f"Wall H={H} m | Hopper H={Hh} m</sup>"
                ),
                scene=dict(
                    xaxis_title="Length (m)",
                    yaxis_title="Width (m)",
                    zaxis_title="Height (m)",
                    aspectmode="data"
                ),
                height=650,
                showlegend=False
            )

            fig.show(renderer="colab")

        except Exception as e:
            print("3D MODEL ERROR:", e)


# Refresh button
refresh_button = widgets.Button(
    description="Refresh 3D Model",
    button_style="success",
    icon="refresh"
)

refresh_button.on_click(draw_dynamic_bunker)

# Automatically redraw when geometry changes
geometry_widgets = [
    L_outer, W_outer, H_wall, t_wall,
    H_hopper, L_outlet, W_outlet
]

for control in geometry_widgets:
    control.observe(draw_dynamic_bunker, names="value")

display(refresh_button, bunker_3d_out)

# Draw initial model
draw_dynamic_bunker()


Button(button_style='success', description='Refresh 3D Model', icon='refresh', style=ButtonStyle())

Output()

In [ ]:

# ==========================================
# DYNAMIC VERTICAL WALL STRUCTURAL ANALYSIS
# ==========================================
with analysis_output:
    clear_output(wait=True)
import numpy as np
import pandas as pd
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display, clear_output

# ------------------------------------------
# SUPPORT CONDITION INPUT
# ------------------------------------------

support_condition = widgets.Dropdown(
    options=[
        ("Cantilever", "cantilever"),
        ("Simply supported", "simply"),
        ("Fixed at both ends", "fixed")
    ],
    value="cantilever",
    description="Wall support:"
)

analysis_output = widgets.Output()

analyze_button = widgets.Button(
    description="Update Structural Analysis",
    button_style="success",
    icon="calculator"
)


# ------------------------------------------
# WALL ANALYSIS CALCULATION
# ------------------------------------------

def perform_wall_analysis(H, q, support):

    if H <= 0 or q < 0:
        raise ValueError("Invalid wall height or pressure.")

    b = 1.0  # One metre wide wall strip

    x = np.linspace(0, H, 150)

    if support == "cantilever":

        V = q * (H - x) * b
        M = q * (H - x)**2 * b / 2

        R_bottom = q * H * b
        R_top = 0

        M_max = q * H**2 * b / 2
        M_min = 0

    elif support == "simply":

        V = q * (H/2 - x) * b
        M = q * x * (H-x) * b / 2

        R_bottom = q * H * b / 2
        R_top = q * H * b / 2

        M_max = q * H**2 * b / 8
        M_min = 0

    elif support == "fixed":

        V = q * (H/2 - x) * b

        M = (
            -q * H**2 / 12
            + q * H * x / 2
            - q * x**2 / 2
        ) * b

        R_bottom = q * H * b / 2
        R_top = q * H * b / 2

        M_max = q * H**2 * b / 24
        M_min = -q * H**2 * b / 12

    return {
        "x": x,
        "V": V,
        "M": M,
        "R_bottom": R_bottom,
        "R_top": R_top,
        "M_max": M_max,
        "M_min": M_min
    }


# ------------------------------------------
# AUTOMATIC STRUCTURAL ANALYSIS
# ------------------------------------------

def run_structural_analysis(change=None):

    with analysis_output:

        clear_output(wait=True)

        try:

            # Read latest original inputs
            inp = get_inputs()

            H = inp["H_wall"]

            # Screening pressure from original calculation module
            stage1_results = calculate_stage1(inp)

            q = stage1_results["p_screen_base"]

            support = support_condition.value

            result = perform_wall_analysis(H, q, support)

            print("RECTANGULAR RCC BUNKER")
            print("VERTICAL WALL STRUCTURAL ANALYSIS")
            print("-----------------------------------")

            print(f"Outer length = {inp['L_outer']:.2f} m")
            print(f"Outer width = {inp['W_outer']:.2f} m")
            print(f"Wall thickness = {inp['t_wall']:.2f} m")
            print(f"Wall height = {H:.2f} m")
            print(f"Hopper height = {inp['H_hopper']:.2f} m")
            print(f"Screening pressure = {q:.2f} kN/m²")
            print(f"Support condition = {support}")

            # ----------------------------------
            # SUPPORT REACTIONS
            # ----------------------------------

            print("\nSUPPORT REACTIONS")

            print(
                f"Bottom reaction = "
                f"{result['R_bottom']:.2f} kN"
            )

            print(
                f"Top reaction = "
                f"{result['R_top']:.2f} kN"
            )

            # ----------------------------------
            # BENDING MOMENTS
            # ----------------------------------

            print("\nBENDING MOMENTS")

            print(
                f"Maximum positive moment = "
                f"{result['M_max']:.2f} kN-m"
            )

            print(
                f"Maximum negative moment = "
                f"{result['M_min']:.2f} kN-m"
            )

            # ----------------------------------
            # MAXIMUM SHEAR FORCE
            # ----------------------------------

            print("\nMAXIMUM SHEAR FORCE")

            print(
                f"{np.max(np.abs(result['V'])):.2f} kN"
            )

            # ----------------------------------
            # RESULTS TABLE
            # ----------------------------------

            df = pd.DataFrame({
                "Height from bottom (m)": result["x"],
                "Shear force (kN)": result["V"],
                "Bending moment (kN-m)": result["M"]
            })

            print("\nSTRUCTURAL ANALYSIS RESULTS")

            display(df.round(3).iloc[::15])

            # ==================================
            # SHEAR FORCE DIAGRAM (SFD)
            # ==================================

            fig1 = go.Figure()

            fig1.add_trace(go.Scatter(
                x=result["V"],
                y=result["x"],
                mode="lines",
                line=dict(width=3),
                fill="tozerox",
                name="Shear Force"
            ))

            fig1.add_vline(
                x=0,
                line_dash="dash",
                line_width=2
            )

            fig1.update_layout(
                title="SHEAR FORCE DIAGRAM (SFD)",
                xaxis_title="Shear Force (kN)",
                yaxis_title="Height from bottom (m)",
                height=500,
                width=750,
                template="plotly_white"
            )

            display(fig1)

            # ==================================
            # BENDING MOMENT DIAGRAM (BMD)
            # ==================================

            fig2 = go.Figure()

            fig2.add_trace(go.Scatter(
                x=result["M"],
                y=result["x"],
                mode="lines",
                line=dict(width=3),
                fill="tozerox",
                name="Bending Moment"
            ))

            # Zero reference line
            fig2.add_vline(
                x=0,
                line_dash="dash",
                line_width=2
            )

            fig2.update_layout(
                title="BENDING MOMENT DIAGRAM (BMD)",
                xaxis_title="Bending Moment (kN-m)",
                yaxis_title="Height from bottom (m)",
                height=500,
                width=750,
                template="plotly_white",
                showlegend=True
            )

            display(fig2)

        except Exception as e:

            print("STRUCTURAL ANALYSIS ERROR:", e)


# ------------------------------------------
# CONNECT BUTTON
# ------------------------------------------

analyze_button.on_click(run_structural_analysis)


# ------------------------------------------
# AUTOMATIC INPUT UPDATE
# ------------------------------------------

structural_input_widgets = [
    L_outer,
    W_outer,
    H_wall,
    t_wall,
    H_hopper,
    L_outlet,
    W_outlet,
    t_hopper,
    gamma_material,
    K_screen
]

for control in structural_input_widgets:

    control.observe(
        run_structural_analysis,
        names="value"
    )


# Update when support condition changes
support_condition.observe(
    run_structural_analysis,
    names="value"
)


# ------------------------------------------
# DISPLAY STRUCTURAL ANALYSIS
# ------------------------------------------

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Dynamic Vertical Wall Structural Analysis</h3>"
        ),

        support_condition,

        analyze_button,

        analysis_output
    ])
)


# Initial analysis
run_structural_analysis()


In [ ]:

# ==========================================================
# STEP 2: HOPPER SLAB STRUCTURAL ANALYSIS
# MATPLOTLIB VERSION - CORRECTED SFD AND BMD
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

# Output area
hopper_analysis_output = widgets.Output()


def run_hopper_analysis(change=None):

    with hopper_analysis_output:

        clear_output(wait=True)

        try:

            # ------------------------------------------
            # 1. GET ORIGINAL INPUT VALUES
            # ------------------------------------------

            inp = get_inputs()

            L = float(inp["L_outer"])
            W = float(inp["W_outer"])
            t = float(inp["t_wall"])

            H_wall_value = float(inp["H_wall"])
            Hh = float(inp["H_hopper"])

            Lo = float(inp["L_outlet"])
            Wo = float(inp["W_outlet"])

            gamma = float(inp["gamma_material"])
            K = float(inp["K_screen"])

            # ------------------------------------------
            # 2. INTERNAL DIMENSIONS
            # ------------------------------------------

            Li = L - 2*t
            Wi = W - 2*t

            if min(L, W, t, H_wall_value, Hh, Lo, Wo, gamma) <= 0:
                raise ValueError(
                    "All dimensions and unit weights must be positive."
                )

            if Li <= 0 or Wi <= 0:
                raise ValueError(
                    "Wall thickness is too large."
                )

            if Lo >= Li or Wo >= Wi:
                raise ValueError(
                    "Outlet dimensions must be smaller than internal dimensions."
                )

            if K < 0:
                raise ValueError(
                    "Screening coefficient cannot be negative."
                )

            # ------------------------------------------
            # 3. FOUR HOPPER FACES
            # ------------------------------------------

            faces = [
                {
                    "Face": "Longitudinal face 1",
                    "Run": (Wi-Wo)/2,
                    "Panel width": (Li+Lo)/2
                },
                {
                    "Face": "Longitudinal face 2",
                    "Run": (Wi-Wo)/2,
                    "Panel width": (Li+Lo)/2
                },
                {
                    "Face": "Transverse face 1",
                    "Run": (Li-Lo)/2,
                    "Panel width": (Wi+Wo)/2
                },
                {
                    "Face": "Transverse face 2",
                    "Run": (Li-Lo)/2,
                    "Panel width": (Wi+Wo)/2
                }
            ]

            results = []
            diagram_data = []

            # ------------------------------------------
            # 4. ANALYSE EACH HOPPER FACE
            # ------------------------------------------

            for face in faces:

                run = face["Run"]
                panel_width = face["Panel width"]

                if run <= 0 or panel_width <= 0:
                    raise ValueError(
                        "Invalid hopper face geometry."
                    )

                # Sloping length
                S = np.sqrt(Hh**2 + run**2)

                # Slope angle from horizontal
                theta = np.arctan2(Hh, run)
                theta_deg = np.degrees(theta)

                # --------------------------------------
                # 5. NORMAL PRESSURE
                # --------------------------------------

                pv_top = gamma * H_wall_value
                pv_bottom = gamma * (H_wall_value + Hh)

                ph_top = K * pv_top
                ph_bottom = K * pv_bottom

                q_top = (
                    pv_top * np.cos(theta)
                    + ph_top * np.sin(theta)
                )

                q_bottom = (
                    pv_bottom * np.cos(theta)
                    + ph_bottom * np.sin(theta)
                )

                # --------------------------------------
                # 6. VARYING LOAD ON ONE-METRE STRIP
                # --------------------------------------

                s = np.linspace(0, S, 1001)

                q = q_top + (q_bottom-q_top)*s/S

                # Load in kN/m for one-metre strip
                load = q.copy()

                ds = np.diff(s)

                # --------------------------------------
                # 7. TOTAL LOAD AND SUPPORT REACTIONS
                # --------------------------------------

                load_segments = (
                    (load[:-1] + load[1:]) / 2
                ) * ds

                total_load = np.sum(load_segments)

                load_moment_segments = (
                    (load[:-1]*s[:-1] + load[1:]*s[1:]) / 2
                ) * ds

                total_load_moment = np.sum(
                    load_moment_segments
                )

                R_right = total_load_moment / S
                R_left = total_load - R_right

                # --------------------------------------
                # 8. SHEAR FORCE CALCULATION
                # --------------------------------------

                cumulative_load = np.zeros(len(s))

                cumulative_load[1:] = np.cumsum(
                    load_segments
                )

                V = R_left - cumulative_load

                # --------------------------------------
                # 9. BENDING MOMENT CALCULATION
                # --------------------------------------

                moment_segments = (
                    (V[:-1] + V[1:]) / 2
                ) * ds

                M = np.zeros(len(s))

                M[1:] = np.cumsum(
                    moment_segments
                )

                # --------------------------------------
                # 10. MAXIMUM VALUES
                # --------------------------------------

                max_shear = np.max(np.abs(V))
                max_moment = np.max(np.abs(M))

                max_moment_position = s[
                    np.argmax(np.abs(M))
                ]

                results.append({
                    "Hopper face": face["Face"],
                    "Slope angle (deg)": round(theta_deg, 2),
                    "Sloping length (m)": round(S, 3),
                    "Panel width (m)": round(panel_width, 3),
                    "Top pressure (kN/m2)": round(q_top, 3),
                    "Bottom pressure (kN/m2)": round(q_bottom, 3),
                    "Left reaction (kN/m)": round(R_left, 3),
                    "Right reaction (kN/m)": round(R_right, 3),
                    "Maximum shear (kN/m)": round(max_shear, 3),
                    "Maximum moment (kN.m/m)": round(max_moment, 3)
                })

                diagram_data.append({
                    "Face": face["Face"],
                    "Distance": s,
                    "Shear": V,
                    "Moment": M
                })

            # ------------------------------------------
            # 11. DISPLAY INPUT DATA
            # ------------------------------------------

            print("RECTANGULAR RCC HOPPER SLAB ANALYSIS")
            print("="*55)

            print(f"Internal length : {Li:.3f} m")
            print(f"Internal width  : {Wi:.3f} m")
            print(f"Wall height     : {H_wall_value:.3f} m")
            print(f"Hopper height   : {Hh:.3f} m")
            print(f"Outlet length   : {Lo:.3f} m")
            print(f"Outlet width    : {Wo:.3f} m")

            # ------------------------------------------
            # 12. RESULTS TABLE
            # ------------------------------------------

            df_hopper = pd.DataFrame(results)

            print("\nHOPPER FACE ANALYSIS RESULTS")

            display(df_hopper)

            print("\nGOVERNING PRELIMINARY RESULTS")

            print(
                "Maximum shear force =",
                round(df_hopper["Maximum shear (kN/m)"].max(), 3),
                "kN/m"
            )

            print(
                "Maximum bending moment =",
                round(df_hopper["Maximum moment (kN.m/m)"].max(), 3),
                "kN.m/m"
            )

            # ------------------------------------------
            # 13. SHEAR FORCE DIAGRAM - MATPLOTLIB
            # ------------------------------------------

            print("\nSHEAR FORCE DIAGRAM (SFD)")

            fig1, ax1 = plt.subplots(figsize=(10, 5))

            for item in diagram_data:

                ax1.plot(
                    item["Distance"],
                    item["Shear"],
                    linewidth=2,
                    label=item["Face"]
                )

            ax1.axhline(
                y=0,
                linestyle="--",
                linewidth=1
            )

            ax1.set_title(
                "HOPPER SLAB SHEAR FORCE DIAGRAM"
            )

            ax1.set_xlabel(
                "Distance along inclined face (m)"
            )

            ax1.set_ylabel(
                "Shear force (kN/m)"
            )

            ax1.grid(True)
            ax1.legend()
            fig1.tight_layout()

            display(fig1)
            plt.close(fig1)

            # ------------------------------------------
            # 14. BENDING MOMENT DIAGRAM - MATPLOTLIB
            # ------------------------------------------

            print("\nBENDING MOMENT DIAGRAM (BMD)")

            fig2, ax2 = plt.subplots(figsize=(10, 5))

            for item in diagram_data:

                ax2.plot(
                    item["Distance"],
                    item["Moment"],
                    linewidth=2,
                    label=item["Face"]
                )

            ax2.axhline(
                y=0,
                linestyle="--",
                linewidth=1
            )

            ax2.set_title(
                "HOPPER SLAB BENDING MOMENT DIAGRAM"
            )

            ax2.set_xlabel(
                "Distance along inclined face (m)"
            )

            ax2.set_ylabel(
                "Bending moment (kN.m/m)"
            )

            ax2.grid(True)
            ax2.legend()
            fig2.tight_layout()

            display(fig2)
            plt.close(fig2)

            # ------------------------------------------
            # 15. ENGINEERING NOTE
            # ------------------------------------------

            print("\nENGINEERING NOTE")
            print("-"*50)

            print(
                "This is a preliminary one-metre-wide "
                "simply supported strip analysis."
            )

            print(
                "Pressure values are screening estimates, "
                "not verified IS 4995 design pressures."
            )

            print(
                "Actual hopper slab design must consider "
                "plate action, edge restraint, flow conditions, "
                "support forces and applicable load combinations."
            )

            print(
                "Do not use these results directly for "
                "final RCC reinforcement or construction."
            )

        except Exception as e:

            print("HOPPER ANALYSIS ERROR:")
            print(str(e))


# ==========================================================
# 16. BUTTON AND AUTOMATIC UPDATES
# ==========================================================

hopper_button = widgets.Button(
    description="Calculate Hopper Slab",
    button_style="success",
    icon="calculator"
)

hopper_button.on_click(run_hopper_analysis)

# Original input widgets
hopper_input_widgets = [
    L_outer,
    W_outer,
    H_wall,
    t_wall,
    H_hopper,
    L_outlet,
    W_outlet,
    gamma_material,
    K_screen
]

# Avoid duplicate callback registration
for control in hopper_input_widgets:

    try:
        control.unobserve(
            run_hopper_analysis,
            names="value"
        )
    except (ValueError, KeyError):
        pass

    control.observe(
        run_hopper_analysis,
        names="value"
    )

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Dynamic Hopper Slab Structural Analysis</h3>"
        ),
        hopper_button,
        hopper_analysis_output
    ])
)

# Initial calculation
run_hopper_analysis()


In [ ]:

# ==========================================================
# STEP 3: FOUR-BEAM OUTLET AND CORNER-COLUMN ANALYSIS
# ==========================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

outlet_output = widgets.Output()

# Additional structural inputs
outlet_beam_width = widgets.FloatText(
    value=300, description="Beam width (mm)"
)
outlet_beam_depth = widgets.FloatText(
    value=450, description="Beam depth (mm)"
)
column_width = widgets.FloatText(
    value=300, description="Column width (mm)"
)
column_depth = widgets.FloatText(
    value=300, description="Column depth (mm)"
)
column_height = widgets.FloatText(
    value=3.0, description="Column height (m)"
)

# Preliminary load factor; editable
outlet_load_factor = widgets.FloatText(
    value=1.5, description="Preliminary load factor"
)

hopper_face_actions = []
outlet_beam_results = []
corner_column_results = []


def calculate_hopper_face_actions(inp):

    L = float(inp["L_outer"])
    W = float(inp["W_outer"])
    t = float(inp["t_wall"])
    H = float(inp["H_wall"])
    Hh = float(inp["H_hopper"])
    Lo = float(inp["L_outlet"])
    Wo = float(inp["W_outlet"])

    gamma = float(inp["gamma_material"])
    gamma_c = float(inp["gamma_concrete"])
    K = float(inp["K_screen"])
    th = float(inp["t_hopper"])

    Li = L - 2*t
    Wi = W - 2*t

    if min(Li, Wi, Lo, Wo, H, Hh, gamma, gamma_c, th) <= 0:
        raise ValueError("Check all bunker dimensions and material properties.")

    if Lo >= Li or Wo >= Wi:
        raise ValueError("Outlet dimensions must be smaller than internal dimensions.")

    if K < 0:
        raise ValueError("Screening coefficient cannot be negative.")

    faces = [
        {"Face": "Longitudinal beam 1",
         "Run": (Wi-Wo)/2, "Outlet edge": Lo},

        {"Face": "Longitudinal beam 2",
         "Run": (Wi-Wo)/2, "Outlet edge": Lo},

        {"Face": "Transverse beam 1",
         "Run": (Li-Lo)/2, "Outlet edge": Wo},

        {"Face": "Transverse beam 2",
         "Run": (Li-Lo)/2, "Outlet edge": Wo}
    ]

    actions = []

    for face in faces:

        run = face["Run"]
        edge = face["Outlet edge"]

        if run <= 0 or edge <= 0:
            raise ValueError("Invalid hopper face geometry.")

        S = np.sqrt(Hh**2 + run**2)
        theta = np.arctan2(Hh, run)

        # Screening pressure estimate
        pv_top = gamma*H
        pv_bottom = gamma*(H+Hh)

        ph_top = K*pv_top
        ph_bottom = K*pv_bottom

        q_top = (
            pv_top*np.cos(theta)
            + ph_top*np.sin(theta)
        )

        q_bottom = (
            pv_bottom*np.cos(theta)
            + ph_bottom*np.sin(theta)
        )

        # Hopper slab self-weight normal component
        q_self = gamma_c*th*np.cos(theta)

        q_top += q_self
        q_bottom += q_self

        s = np.linspace(0, S, 1001)
        q = q_top + (q_bottom-q_top)*s/S

        ds = np.diff(s)

        load_segments = (
            (q[:-1]+q[1:])/2
        )*ds

        total_load = np.sum(load_segments)

        moment_segments = (
            (q[:-1]*s[:-1]+q[1:]*s[1:])/2
        )*ds

        total_moment = np.sum(moment_segments)

        # Reactions for one-metre-wide strip
        R_outlet = total_moment/S
        R_top = total_load-R_outlet

        cumulative_load = np.zeros(len(s))
        cumulative_load[1:] = np.cumsum(load_segments)

        V = R_top-cumulative_load

        M = np.zeros(len(s))
        M[1:] = np.cumsum(
            ((V[:-1]+V[1:])/2)*ds
        )

        actions.append({
            "Face": face["Face"],
            "Run": run,
            "Slope length": S,
            "Outlet edge": edge,
            "Angle": np.degrees(theta),
            "Top pressure": q_top,
            "Bottom pressure": q_bottom,
            "Outlet reaction": R_outlet,
            "Top reaction": R_top,
            "Max moment": np.max(np.abs(M)),
            "Max shear": np.max(np.abs(V))
        })

    return actions


def run_outlet_analysis(change=None):

    global hopper_face_actions
    global outlet_beam_results
    global corner_column_results

    with outlet_output:
        clear_output(wait=True)

        try:

            inp = get_inputs()

            bw = outlet_beam_width.value/1000
            bd = outlet_beam_depth.value/1000

            cw = column_width.value/1000
            cd = column_depth.value/1000
            ch = column_height.value

            LF = outlet_load_factor.value

            if min(bw, bd, cw, cd, ch, LF) <= 0:
                raise ValueError("Beam, column and load-factor values must be positive.")

            if bd <= bw:
                raise ValueError("Beam depth must exceed beam width.")

            if bd <= 0 or cd <= 0:
                raise ValueError("Invalid beam or column dimensions.")

            gamma_c = float(inp["gamma_concrete"])

            hopper_face_actions = calculate_hopper_face_actions(inp)

            # Beam self-weight
            beam_self_weight = gamma_c*bw*bd

            # Each beam receives the corresponding hopper outlet reaction
            definitions = [
                ("Longitudinal beam 1",
                 inp["L_outlet"], hopper_face_actions[0]["Outlet reaction"]),

                ("Longitudinal beam 2",
                 inp["L_outlet"], hopper_face_actions[1]["Outlet reaction"]),

                ("Transverse beam 1",
                 inp["W_outlet"], hopper_face_actions[2]["Outlet reaction"]),

                ("Transverse beam 2",
                 inp["W_outlet"], hopper_face_actions[3]["Outlet reaction"])
            ]

            outlet_beam_results = []

            for name, span, hopper_load in definitions:

                # Service load
                service_w = hopper_load+beam_self_weight

                # Preliminary factored load
                factored_w = LF*service_w

                # Simply supported beam
                R = factored_w*span/2
                Vmax = R
                Mmax = factored_w*span**2/8

                outlet_beam_results.append({
                    "Beam": name,
                    "Span (m)": span,
                    "Service load (kN/m)": service_w,
                    "Factored load (kN/m)": factored_w,
                    "Reaction at each end (kN)": R,
                    "Maximum shear (kN)": Vmax,
                    "Maximum moment (kN.m)": Mmax
                })

            # --------------------------------------
            # CORNER COLUMN REACTIONS
            # Beam order:
            # L1, L2, T1, T2
            # --------------------------------------

            L1 = outlet_beam_results[0]["Reaction at each end (kN)"]
            L2 = outlet_beam_results[1]["Reaction at each end (kN)"]
            T1 = outlet_beam_results[2]["Reaction at each end (kN)"]
            T2 = outlet_beam_results[3]["Reaction at each end (kN)"]

            corner_column_results = [
                {"Column": "Corner column 1", "Axial load": L1+T1},
                {"Column": "Corner column 2", "Axial load": L1+T2},
                {"Column": "Corner column 3", "Axial load": L2+T2},
                {"Column": "Corner column 4", "Axial load": L2+T1}
            ]

            # --------------------------------------
            # DISPLAY
            # --------------------------------------

            print("FOUR-BEAM OUTLET ANALYSIS")
            print("="*55)

            print("\nHOPPER OUTLET REACTIONS")

            face_df = pd.DataFrame([
                {
                    "Face": a["Face"],
                    "Outlet reaction (kN/m)": a["Outlet reaction"],
                    "Top reaction (kN/m)": a["Top reaction"]
                }
                for a in hopper_face_actions
            ])

            display(face_df.round(3))

            print("\nSUPPORTING BEAM RESULTS")
            display(pd.DataFrame(outlet_beam_results).round(3))

            print("\nCORNER COLUMN AXIAL REACTIONS")
            display(pd.DataFrame(corner_column_results).round(3))

            print("\nADDITIONAL INPUTS")
            print(f"Beam size = {bw*1000:.0f} x {bd*1000:.0f} mm")
            print(f"Column size = {cw*1000:.0f} x {cd*1000:.0f} mm")
            print(f"Column height = {ch:.2f} m")
            print(f"Preliminary load factor = {LF:.2f}")

            print("\nSUPPORT MODEL")
            print("Four simply supported beams with pinned end connections.")
            print("Beam end moments are zero in this idealised model.")

            print("\nLIMITATION")
            print("Column values are axial reactions only, not column capacity checks.")
            print("The pressure model is a screening estimate, not an IS 4995 calculation.")

        except Exception as e:
            print("OUTLET ANALYSIS ERROR:", str(e))


outlet_button = widgets.Button(
    description="Calculate Outlet and Beams",
    button_style="success",
    icon="calculator"
)

outlet_button.on_click(run_outlet_analysis)

step3_widgets = [
    L_outer, W_outer, H_wall, t_wall,
    H_hopper, L_outlet, W_outlet, t_hopper,
    gamma_material, gamma_concrete, K_screen,
    outlet_beam_width, outlet_beam_depth,
    column_width, column_depth, column_height,
    outlet_load_factor
]

for control in step3_widgets:
    try:
        control.unobserve(run_outlet_analysis, names="value")
    except (ValueError, KeyError):
        pass
    control.observe(run_outlet_analysis, names="value")

display(widgets.VBox([
    widgets.HTML("<h3>Step 3: Four-Beam Outlet and Column Analysis</h3>"),
    outlet_beam_width,
    outlet_beam_depth,
    column_width,
    column_depth,
    column_height,
    outlet_load_factor,
    outlet_button,
    outlet_output
]))

run_outlet_analysis()


In [ ]:

# ==========================================================
# STEP 4: RCC REINFORCEMENT DESIGN
# ==========================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

reinforcement_output = widgets.Output()

slab_main_dia = widgets.Dropdown(
    options=[8, 10, 12, 16, 20],
    value=12,
    description="Slab main bar (mm)"
)

slab_distribution_dia = widgets.Dropdown(
    options=[8, 10, 12, 16],
    value=10,
    description="Slab distribution bar (mm)"
)

beam_main_dia = widgets.Dropdown(
    options=[12, 16, 20, 25, 32],
    value=16,
    description="Beam main bar (mm)"
)

stirrup_dia = widgets.Dropdown(
    options=[8, 10, 12],
    value=8,
    description="Beam stirrup bar (mm)"
)

# User must enter actual available anchorage length
available_anchorage = widgets.FloatText(
    value=0,
    description="Available anchorage (mm)",
    style={"description_width": "initial"}
)


def bar_area(dia):
    return np.pi*dia**2/4


def required_flexural_steel(Mu_kNm, b, d, fck, fy):

    Mu = Mu_kNm*1e6

    if Mu < 0 or min(b, d, fck, fy) <= 0:
        raise ValueError("Invalid flexural design values.")

    alpha = 0.42*0.87*fy/(0.36*fck*b)

    disc = d**2-4*alpha*Mu/(0.87*fy)

    if disc < 0:
        raise ValueError(
            "Moment exceeds singly reinforced section capacity. "
            "Increase section size or design as doubly reinforced."
        )

    Ast = (
        d-np.sqrt(disc)
    )/(2*alpha)

    return Ast


def limiting_steel_ratio(fy):

    if fy <= 250:
        return 0.53
    elif fy <= 415:
        return 0.48
    else:
        return 0.46


def run_reinforcement_design(change=None):

    global slab_reinforcement_results
    global beam_reinforcement_results

    with reinforcement_output:
        clear_output(wait=True)

        try:

            inp = get_inputs()

            if not outlet_beam_results or not hopper_face_actions:
                raise ValueError("Run Step 3 first.")

            fck_value = float(inp["fck"])
            fy_value = float(inp["fy"])
            cover_value = float(inp["cover_mm"])

            slab_thickness = float(inp["t_hopper"])*1000

            dm = float(slab_main_dia.value)
            dd = float(slab_distribution_dia.value)
            db = float(beam_main_dia.value)

            beam_b = float(outlet_beam_width.value)
            beam_D = float(outlet_beam_depth.value)
            stirrup = float(stirrup_dia.value)

            slab_d = slab_thickness-cover_value-dm/2

            beam_d = beam_D-cover_value-stirrup-db/2

            if slab_d <= 0 or beam_d <= 0:
                raise ValueError("Effective depth is not positive. Check cover and section sizes.")

            # --------------------------------------
            # SLAB DESIGN
            # --------------------------------------

            slab_Mu = max(
                a["Max moment"]*1.5
                for a in hopper_face_actions
            )

            Ast_main_req = required_flexural_steel(
                slab_Mu, 1000, slab_d, fck_value, fy_value
            )

            Ast_min = 0.0012*1000*slab_thickness

            Ast_main_req = max(Ast_main_req, Ast_min)

            Ast_dist_req = Ast_min

            spacing_main = min(
                1000*bar_area(dm)/Ast_main_req,
                3*slab_d,
                300
            )

            spacing_dist = min(
                1000*bar_area(dd)/Ast_dist_req,
                5*slab_d,
                450
            )

            Ast_main_prov = 1000*bar_area(dm)/spacing_main
            Ast_dist_prov = 1000*bar_area(dd)/spacing_dist

            slab_reinforcement_results = {
                "Moment": slab_Mu,
                "Effective depth": slab_d,
                "Main required": Ast_main_req,
                "Main provided": Ast_main_prov,
                "Main diameter": dm,
                "Main spacing": spacing_main,
                "Distribution required": Ast_dist_req,
                "Distribution provided": Ast_dist_prov,
                "Distribution diameter": dd,
                "Distribution spacing": spacing_dist
            }

            # --------------------------------------
            # BEAM DESIGN
            # --------------------------------------

            beam_reinforcement_results = []

            for beam in outlet_beam_results:

                Mu = beam["Maximum moment (kN.m)"]

                Ast_req = required_flexural_steel(
                    Mu, beam_b, beam_d, fck_value, fy_value
                )

                Ast_min = 0.85*beam_b*beam_d/fy_value

                Ast_req = max(Ast_req, Ast_min)

                n_bars = max(
                    2,
                    int(np.ceil(Ast_req/bar_area(db)))
                )

                Ast_prov = n_bars*bar_area(db)

                xu = (
                    0.87*fy_value*Ast_prov
                    /(0.36*fck_value*beam_b)
                )

                xu_max = limiting_steel_ratio(fy_value)*beam_d

                Mu_capacity = (
                    0.87*fy_value*Ast_prov
                    *(beam_d-0.42*xu)
                    /1e6
                )

                beam_reinforcement_results.append({
                    "Beam": beam["Beam"],
                    "Moment demand (kN.m)": Mu,
                    "Required steel (mm2)": Ast_req,
                    "Bars": n_bars,
                    "Bar diameter (mm)": db,
                    "Provided steel (mm2)": Ast_prov,
                    "Moment capacity (kN.m)": Mu_capacity,
                    "xu (mm)": xu,
                    "xu max (mm)": xu_max
                })

            # --------------------------------------
            # DISPLAY
            # --------------------------------------

            print("RCC REINFORCEMENT DESIGN")
            print("="*55)

            print("\nHOPPER SLAB REINFORCEMENT")

            print(f"Effective depth = {slab_d:.2f} mm")

            print(
                f"Main reinforcement = {dm:.0f} mm @ "
                f"{spacing_main:.1f} mm c/c"
            )

            print(
                f"Main steel required = {Ast_main_req:.2f} mm2/m"
            )

            print(
                f"Main steel provided = {Ast_main_prov:.2f} mm2/m"
            )

            print(
                f"Distribution reinforcement = {dd:.0f} mm @ "
                f"{spacing_dist:.1f} mm c/c"
            )

            print(
                f"Distribution steel required = {Ast_dist_req:.2f} mm2/m"
            )

            print(
                f"Distribution steel provided = {Ast_dist_prov:.2f} mm2/m"
            )

            print("\nOUTLET BEAM REINFORCEMENT")

            display(
                pd.DataFrame(beam_reinforcement_results).round(3)
            )

            print("\nNOTE:")
            print("Beam top and bottom reinforcement must be detailed separately.")
            print("The displayed bar count is a preliminary tension-steel estimate.")

        except Exception as e:
            print("REINFORCEMENT DESIGN ERROR:", str(e))


reinforcement_button = widgets.Button(
    description="Calculate RCC Reinforcement",
    button_style="success",
    icon="calculator"
)

reinforcement_button.on_click(run_reinforcement_design)

step4_widgets = [
    L_outer, W_outer, H_wall, t_wall,
    H_hopper, L_outlet, W_outlet, t_hopper,
    gamma_material, gamma_concrete, K_screen,
    fck, fy, cover,
    outlet_beam_width, outlet_beam_depth,
    slab_main_dia, slab_distribution_dia,
    beam_main_dia, stirrup_dia
]

for control in step4_widgets:
    try:
        control.unobserve(run_reinforcement_design, names="value")
    except (ValueError, KeyError):
        pass
    control.observe(run_reinforcement_design, names="value")

display(widgets.VBox([
    widgets.HTML("<h3>Step 4: RCC Reinforcement Design</h3>"),
    slab_main_dia,
    slab_distribution_dia,
    beam_main_dia,
    stirrup_dia,
    available_anchorage,
    reinforcement_button,
    reinforcement_output
]))

run_reinforcement_design()


In [ ]:

# ==========================================================
# STEP 5: RCC DESIGN VERIFICATION
# IS 456:2000 - AVAILABLE DESIGN CHECKS
# ==========================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

step5_output = widgets.Output()

# ----------------------------------------------------------
# ADDITIONAL INPUT: ACTUAL BEAM STIRRUP SPACING
# ----------------------------------------------------------

provided_stirrup_spacing = widgets.FloatText(
    value=150,
    description="Stirrup spacing (mm)",
    style={"description_width": "initial"}
)

# Assumed two-legged vertical stirrups
stirrup_legs = widgets.IntText(
    value=2,
    description="Stirrup legs",
    style={"description_width": "initial"}
)

# ----------------------------------------------------------
# IS 456 TABLE 19 DATA
# Rows: percentage of longitudinal tension steel
# Columns: concrete grades M15, M20, M25, M30, M35, M40+
# ----------------------------------------------------------

PT_VALUES = np.array([
    0.15, 0.25, 0.50, 0.75, 1.00, 1.25,
    1.50, 1.75, 2.00, 2.25, 2.50, 2.75, 3.00
])

FCK_GRADES = np.array([15, 20, 25, 30, 35, 40])

TAU_C_TABLE = np.array([
    [0.28, 0.28, 0.29, 0.29, 0.29, 0.30],
    [0.35, 0.36, 0.36, 0.37, 0.37, 0.38],
    [0.46, 0.48, 0.49, 0.50, 0.50, 0.51],
    [0.54, 0.56, 0.57, 0.59, 0.59, 0.60],
    [0.60, 0.62, 0.64, 0.66, 0.67, 0.68],
    [0.64, 0.67, 0.70, 0.71, 0.73, 0.74],
    [0.68, 0.72, 0.74, 0.76, 0.78, 0.79],
    [0.71, 0.75, 0.78, 0.80, 0.82, 0.84],
    [0.71, 0.79, 0.82, 0.84, 0.86, 0.88],
    [0.71, 0.81, 0.85, 0.88, 0.90, 0.92],
    [0.71, 0.82, 0.88, 0.91, 0.93, 0.95],
    [0.71, 0.82, 0.90, 0.94, 0.96, 0.98],
    [0.71, 0.82, 0.92, 0.96, 0.99, 1.01]
])

# IS 456 Table 20 maximum shear stress
TAU_C_MAX_TABLE = np.array([
    2.50, 2.80, 3.10, 3.50, 3.70, 4.00
])


def get_tau_c(pt, fck_value):
    """
    Interpolation in IS 456 Table 19.
    fck must be within M15 to M40.
    """

    if fck_value < 15 or fck_value > 40:
        raise ValueError(
            "IS 456 Table 19 implementation supports "
            "M15 to M40. Check the selected concrete grade."
        )

    pt = max(0.15, min(float(pt), 3.0))

    # Interpolate for percentage steel
    values_at_grades = np.array([
        np.interp(pt, PT_VALUES, TAU_C_TABLE[:, j])
        for j in range(len(FCK_GRADES))
    ])

    # Interpolate for concrete grade
    return float(np.interp(
        fck_value,
        FCK_GRADES,
        values_at_grades
    ))


def get_tau_c_max(fck_value):
    if fck_value < 15 or fck_value > 40:
        raise ValueError("Concrete grade must be between M15 and M40.")

    return float(np.interp(
        fck_value,
        FCK_GRADES,
        TAU_C_MAX_TABLE
    ))


def add_check(checks, name, status, details):
    checks.append({
        "Check": name,
        "Status": status,
        "Details": details
    })


# ----------------------------------------------------------
# MAIN VERIFICATION FUNCTION
# ----------------------------------------------------------

def run_step5_verification(change=None):

    with step5_output:
        clear_output(wait=True)

        try:

            # Check whether previous steps have been executed
            if not outlet_beam_results:
                raise ValueError("Run Step 3 first.")

            if not beam_reinforcement_results:
                raise ValueError("Run Step 4 first.")

            if not isinstance(slab_reinforcement_results, dict):
                raise ValueError("Run Step 4 first.")

            inp = get_inputs()

            fck_value = float(inp["fck"])
            fy_value = float(inp["fy"])
            cover_value = float(inp["cover_mm"])

            beam_b = float(outlet_beam_width.value)
            beam_D = float(outlet_beam_depth.value)

            stirrup_dia_value = float(stirrup_dia.value)
            stirrup_spacing_value = float(
                provided_stirrup_spacing.value
            )
            legs = int(stirrup_legs.value)

            slab_thickness = float(inp["t_hopper"]) * 1000

            if min(
                fck_value, fy_value, beam_b, beam_D,
                stirrup_dia_value, stirrup_spacing_value,
                legs, slab_thickness
            ) <= 0:
                raise ValueError("All design inputs must be positive.")

            if beam_D <= cover_value + stirrup_dia_value:
                raise ValueError("Beam effective depth is invalid.")

            if stirrup_spacing_value <= 0:
                raise ValueError("Enter a valid stirrup spacing.")

            checks = []
            beam_shear_details = []
            beam_flexure_details = []

            # ==================================================
            # A. HOPPER SLAB REINFORCEMENT CHECKS
            # ==================================================

            slab = slab_reinforcement_results

            slab_main_req = float(slab["Main required"])
            slab_main_prov = float(slab["Main provided"])

            slab_dist_req = float(slab["Distribution required"])
            slab_dist_prov = float(slab["Distribution provided"])

            slab_d = float(slab["Effective depth"])

            main_spacing = float(slab["Main spacing"])
            dist_spacing = float(slab["Distribution spacing"])

            main_dia = float(slab["Main diameter"])
            dist_dia = float(slab["Distribution diameter"])

            main_spacing_limit = min(3 * slab_d, 300)
            dist_spacing_limit = min(5 * slab_d, 450)

            # Main reinforcement
            add_check(
                checks,
                "Slab main reinforcement",
                "PASS" if slab_main_prov >= slab_main_req else "FAIL",
                (
                    f"Required = {slab_main_req:.2f} mm2/m; "
                    f"provided = {slab_main_prov:.2f} mm2/m"
                )
            )

            # Distribution reinforcement
            add_check(
                checks,
                "Slab distribution reinforcement",
                "PASS" if slab_dist_prov >= slab_dist_req else "FAIL",
                (
                    f"Required = {slab_dist_req:.2f} mm2/m; "
                    f"provided = {slab_dist_prov:.2f} mm2/m"
                )
            )

            # Main bar spacing
            add_check(
                checks,
                "Slab main bar spacing",
                "PASS" if main_spacing <= main_spacing_limit else "FAIL",
                (
                    f"Provided = {main_spacing:.1f} mm; "
                    f"maximum = {main_spacing_limit:.1f} mm"
                )
            )

            # Distribution bar spacing
            add_check(
                checks,
                "Slab distribution bar spacing",
                "PASS" if dist_spacing <= dist_spacing_limit else "FAIL",
                (
                    f"Provided = {dist_spacing:.1f} mm; "
                    f"maximum = {dist_spacing_limit:.1f} mm"
                )
            )

            # ==================================================
            # B. BEAM FLEXURAL CAPACITY CHECK
            # ==================================================

            for beam, reinforcement in zip(
                outlet_beam_results,
                beam_reinforcement_results
            ):

                Mu = float(
                    beam["Maximum moment (kN.m)"]
                )

                Ast = float(
                    reinforcement["Provided steel (mm2)"]
                )

                xu = float(reinforcement["xu (mm)"])
                xu_max = float(reinforcement["xu max (mm)"])

                Mu_capacity = float(
                    reinforcement["Moment capacity (kN.m)"]
                )

                flexure_status = (
                    "PASS"
                    if Mu_capacity >= Mu and xu <= xu_max
                    else "FAIL"
                )

                beam_flexure_details.append({
                    "Beam": beam["Beam"],
                    "Moment demand (kN.m)": Mu,
                    "Moment capacity (kN.m)": Mu_capacity,
                    "xu (mm)": xu,
                    "xu max (mm)": xu_max,
                    "Provided steel (mm2)": Ast,
                    "Status": flexure_status
                })

            for result in beam_flexure_details:
                add_check(
                    checks,
                    result["Beam"] + " flexural capacity",
                    result["Status"],
                    (
                        f"Mu = {result['Moment demand (kN.m)']:.2f} "
                        f"kN.m; capacity = "
                        f"{result['Moment capacity (kN.m)']:.2f} kN.m; "
                        f"xu = {result['xu (mm)']:.1f} mm; "
                        f"xu,max = {result['xu max (mm)']:.1f} mm"
                    )
                )

            # ==================================================
            # C. BEAM SHEAR DESIGN - IS 456
            # ==================================================

            beam_d = (
                beam_D
                - cover_value
                - stirrup_dia_value
                - float(beam_main_dia.value)/2
            )

            if beam_d <= 0:
                raise ValueError("Beam effective depth is not positive.")

            Asv = (
                legs * np.pi * stirrup_dia_value**2 / 4
            )

            max_spacing_code = min(0.75 * beam_d, 300)

            min_steel_spacing_limit = (
                0.87 * fy_value * Asv
                / (0.4 * beam_b)
            )

            for beam, reinforcement in zip(
                outlet_beam_results,
                beam_reinforcement_results
            ):

                span = float(beam["Span (m)"])
                wu = float(beam["Factored load (kN/m)"])
                Vu_support = float(
                    beam["Maximum shear (kN)"]
                )

                Ast = float(
                    reinforcement["Provided steel (mm2)"]
                )

                # Critical section at distance d from support
                d_m = beam_d / 1000

                Vu_critical = max(
                    0,
                    Vu_support - wu * d_m
                )

                # Percentage longitudinal tension steel
                pt = 100 * Ast / (beam_b * beam_d)

                tau_c = get_tau_c(pt, fck_value)
                tau_c_max = get_tau_c_max(fck_value)

                # Nominal shear stress
                tau_v = (
                    Vu_critical * 1000
                    / (beam_b * beam_d)
                )

                # Concrete shear resistance
                Vc = (
                    tau_c * beam_b * beam_d / 1000
                )

                # Shear to be resisted by stirrups
                Vus = max(0, Vu_critical - Vc)

                # Required stirrup spacing
                if Vus > 0:
                    required_spacing = (
                        0.87 * fy_value * Asv * beam_d
                        / (Vus * 1000)
                    )
                else:
                    required_spacing = float("inf")

                allowable_spacing = min(
                    max_spacing_code,
                    min_steel_spacing_limit,
                    required_spacing
                )

                # Maximum shear stress check
                if tau_v > tau_c_max:
                    shear_status = "FAIL"
                    shear_reason = (
                        "Nominal shear stress exceeds "
                        "maximum concrete shear stress."
                    )

                elif stirrup_spacing_value > max_spacing_code:
                    shear_status = "FAIL"
                    shear_reason = (
                        "Stirrup spacing exceeds maximum "
                        "permitted spacing."
                    )

                elif stirrup_spacing_value > min_steel_spacing_limit:
                    shear_status = "FAIL"
                    shear_reason = (
                        "Minimum shear reinforcement "
                        "requirement is not satisfied."
                    )

                elif stirrup_spacing_value > required_spacing:
                    shear_status = "FAIL"
                    shear_reason = (
                        "Provided stirrup spacing is greater "
                        "than the calculated required spacing."
                    )

                else:
                    shear_status = "PASS"
                    shear_reason = (
                        "Provided stirrup spacing satisfies "
                        "the calculated shear requirements."
                    )

                # Maximum concrete shear stress
                section_status = (
                    "PASS" if tau_v <= tau_c_max else "FAIL"
                )

                beam_shear_details.append({
                    "Beam": beam["Beam"],
                    "Critical shear (kN)": Vu_critical,
                    "pt (%)": pt,
                    "tau_v (N/mm2)": tau_v,
                    "tau_c (N/mm2)": tau_c,
                    "tau_c max (N/mm2)": tau_c_max,
                    "Concrete capacity (kN)": Vc,
                    "Stirrup shear (kN)": Vus,
                    "Required spacing (mm)": (
                        required_spacing
                        if np.isfinite(required_spacing)
                        else max_spacing_code
                    ),
                    "Provided spacing (mm)": stirrup_spacing_value,
                    "Maximum spacing (mm)": max_spacing_code,
                    "Status": shear_status,
                    "Reason": shear_reason
                })

                add_check(
                    checks,
                    beam["Beam"] + " maximum shear stress",
                    section_status,
                    (
                        f"tau_v = {tau_v:.3f} N/mm2; "
                        f"tau_c,max = {tau_c_max:.3f} N/mm2"
                    )
                )

                add_check(
                    checks,
                    beam["Beam"] + " shear reinforcement",
                    shear_status,
                    (
                        f"Required spacing = "
                        f"{required_spacing:.1f} mm; "
                        f"provided = {stirrup_spacing_value:.1f} mm; "
                        f"{shear_reason}"
                    )
                )

            # ==================================================
            # D. DISPLAY RESULTS
            # ==================================================

            results_df = pd.DataFrame(checks)

            print("STEP 5: RCC DESIGN VERIFICATION")
            print("=" * 65)

            print("\nDESIGN INPUTS")
            print(f"Concrete grade = M{fck_value:.0f}")
            print(f"Steel yield strength = {fy_value:.0f} N/mm2")
            print(f"Beam section = {beam_b:.0f} x {beam_D:.0f} mm")
            print(f"Beam effective depth = {beam_d:.1f} mm")
            print(
                f"Stirrups = {legs} legs of "
                f"{stirrup_dia_value:.0f} mm"
            )
            print(
                f"Provided stirrup spacing = "
                f"{stirrup_spacing_value:.1f} mm"
            )

            print("\nDESIGN VERIFICATION SUMMARY")
            display(results_df)

            print("\nBEAM FLEXURAL CAPACITY")
            display(
                pd.DataFrame(beam_flexure_details).round(3)
            )

            print("\nBEAM SHEAR DESIGN")
            display(
                pd.DataFrame(beam_shear_details).round(3)
            )

            # Status totals
            passed = sum(
                1 for x in checks if x["Status"] == "PASS"
            )

            failed = sum(
                1 for x in checks if x["Status"] == "FAIL"
            )

            print("\nOVERALL CHECK SUMMARY")
            print("-" * 45)
            print(f"PASS: {passed}")
            print(f"FAIL: {failed}")

            if failed == 0:
                print("\nAll checks included in this module passed.")
            else:
                print(
                    "\nSome checks failed. Revise the design "
                    "and run Step 5 again."
                )

            print("\nDESIGN BASIS AND LIMITATIONS")
            print(
                "1. Slab reinforcement checks use the reinforcement "
                "requirements calculated in Step 4."
            )
            print(
                "2. Beam flexural checks use the moment demand and "
                "steel area calculated in Step 4."
            )
            print(
                "3. Beam shear uses IS 456 Table 19 and Table 20, "
                "with a critical section at effective depth from support."
            )
            print(
                "4. Two-legged or user-specified stirrup legs are "
                "assumed to be vertical closed stirrups."
            )
            print(
                "5. This module does not check torsion, crack width, "
                "deflection, column capacity or two-way slab action."
            )
            print(
                "6. The accuracy of all checks depends on the "
                "accuracy of the load model in Steps 3 and 4."
            )
            print(
                "7. This is a preliminary software verification, "
                "not approval for construction."
            )

        except Exception as e:
            print("STEP 5 ERROR:", str(e))


# ----------------------------------------------------------
# BUTTON AND AUTOMATIC UPDATE
# ----------------------------------------------------------

step5_button = widgets.Button(
    description="Run Step 5 Design Verification",
    button_style="success",
    icon="check"
)

step5_button.on_click(run_step5_verification)

step5_controls = [
    provided_stirrup_spacing,
    stirrup_legs,
    slab_main_dia,
    slab_distribution_dia,
    beam_main_dia,
    stirrup_dia,
    outlet_beam_width,
    outlet_beam_depth,
    fck,
    fy,
    cover
]

for control in step5_controls:
    try:
        control.unobserve(
            run_step5_verification,
            names="value"
        )
    except (ValueError, KeyError):
        pass

    control.observe(
        run_step5_verification,
        names="value"
    )


display(widgets.VBox([
    widgets.HTML(
        "<h3>Step 5: RCC Design Verification - IS 456</h3>"
    ),
    provided_stirrup_spacing,
    stirrup_legs,
    step5_button,
    step5_output
]))

run_step5_verification()


In [ ]:

# ==========================================================
# STEP 6: FINAL DESIGN REPORT AND PDF DOWNLOAD
# ==========================================================

# 1. INSTALL REPORTLAB AUTOMATICALLY

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("reportlab") is None:
    print("Installing ReportLab...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "reportlab"
    ])

print("ReportLab is ready.")

# 2. IMPORT LIBRARIES

import os
import numpy as np
import pandas as pd
import ipywidgets as widgets

from datetime import datetime
from html import escape

from IPython.display import display, clear_output

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle
)

# ==========================================================
# REPORT OUTPUT
# ==========================================================

step6_output = widgets.Output()

report_filename = (
    "/content/Rectangular_RCC_Hopper_Bunker_Design_Report.pdf"
)

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

def create_report_table(data, widths=None):

    table = Table(
        data,
        colWidths=widths,
        repeatRows=1
    )

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0),
         colors.HexColor("#24476B")),

        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),

        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),

        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),

        ("FONTSIZE", (0, 0), (-1, -1), 7),

        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),

        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),

        ("ROWBACKGROUNDS", (0, 1), (-1, -1),
         [colors.white, colors.HexColor("#EAF0F5")]),

        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5)
    ]))

    return table


def dataframe_to_data(df):

    if df is None or len(df) == 0:
        return [["No results available"]]

    data = [list(df.columns)]

    for row in df.itertuples(index=False, name=None):

        formatted_row = []

        for value in row:

            if pd.isna(value):
                formatted_row.append("-")

            elif isinstance(value, (float, np.floating)):
                formatted_row.append(f"{value:.2f}")

            else:
                formatted_row.append(str(value))

        data.append(formatted_row)

    return data


# ==========================================================
# MAIN PDF GENERATION FUNCTION
# ==========================================================

def generate_step6_report(change=None):

    with step6_output:

        clear_output(wait=True)

        try:

            # ----------------------------------------------
            # CHECK PREVIOUS STEPS
            # ----------------------------------------------

            if not outlet_beam_results:
                raise ValueError("Please run Step 3 first.")

            if not hopper_face_actions:
                raise ValueError("Hopper analysis results are missing.")

            if not beam_reinforcement_results:
                raise ValueError("Please run Step 4 first.")

            if not isinstance(slab_reinforcement_results, dict):
                raise ValueError("Slab reinforcement results are missing.")

            inp = get_inputs()

            # ----------------------------------------------
            # CREATE PDF DOCUMENT
            # ----------------------------------------------

            doc = SimpleDocTemplate(
                report_filename,
                pagesize=A4,
                rightMargin=15*mm,
                leftMargin=15*mm,
                topMargin=15*mm,
                bottomMargin=18*mm
            )

            styles = getSampleStyleSheet()

            styles.add(ParagraphStyle(
                name="MainTitle",
                parent=styles["Title"],
                fontSize=17,
                leading=22,
                alignment=1,
                textColor=colors.HexColor("#24476B"),
                spaceAfter=12
            ))

            styles.add(ParagraphStyle(
                name="SectionTitle",
                parent=styles["Heading2"],
                fontSize=12,
                leading=15,
                textColor=colors.HexColor("#24476B"),
                spaceBefore=12,
                spaceAfter=7
            ))

            styles.add(ParagraphStyle(
                name="ReportNote",
                parent=styles["Normal"],
                fontSize=8,
                leading=11
            ))

            story = []

            # ==================================================
            # 1. TITLE
            # ==================================================

            story.append(Paragraph(
                "RECTANGULAR RCC HOPPER BUNKER",
                styles["MainTitle"]
            ))

            story.append(Paragraph(
                "Structural Analysis and Reinforcement Design Report",
                styles["Heading2"]
            ))

            story.append(Spacer(1, 8))

            story.append(Paragraph(
                "Report generated: " +
                datetime.now().strftime("%d-%m-%Y %H:%M"),
                styles["Normal"]
            ))

            # ==================================================
            # 2. DESIGN INPUTS
            # ==================================================

            story.append(Paragraph(
                "1. GENERAL DESIGN INPUTS",
                styles["SectionTitle"]
            ))

            geometry_data = [
                ["Parameter", "Value"],
                ["Outer length", f"{inp['L_outer']} m"],
                ["Outer width", f"{inp['W_outer']} m"],
                ["Vertical wall height", f"{inp['H_wall']} m"],
                ["Wall thickness", f"{inp['t_wall']} m"],
                ["Hopper height", f"{inp['H_hopper']} m"],
                ["Outlet length", f"{inp['L_outlet']} m"],
                ["Outlet width", f"{inp['W_outlet']} m"],
                ["Hopper slab thickness", f"{inp['t_hopper']} m"],
                ["Bulk material unit weight",
                 f"{inp['gamma_material']} kN/m3"],
                ["Concrete unit weight",
                 f"{inp['gamma_concrete']} kN/m3"],
                ["Concrete grade", f"M{inp['fck']}"],
                ["Steel grade", f"Fe {inp['fy']}"],
                ["Nominal cover", f"{inp['cover_mm']} mm"]
            ]

            story.append(create_report_table(
                geometry_data,
                [85*mm, 85*mm]
            ))

            # ==================================================
            # 3. HOPPER FACE ANALYSIS
            # ==================================================

            story.append(Paragraph(
                "2. HOPPER FACE ANALYSIS",
                styles["SectionTitle"]
            ))

            story.append(Paragraph(
                "Results obtained from Step 3. The pressure model "
                "currently uses a screening estimate and is not "
                "a verified IS 4995 pressure calculation.",
                styles["ReportNote"]
            ))

            face_df = pd.DataFrame([
                {
                    "Face": x["Face"],
                    "Run (m)": x["Run"],
                    "Slope (m)": x["Slope length"],
                    "Angle": x["Angle"],
                    "Top reaction": x["Top reaction"],
                    "Outlet reaction": x["Outlet reaction"],
                    "Moment": x["Max moment"],
                    "Shear": x["Max shear"]
                }
                for x in hopper_face_actions
            ])

            story.append(Spacer(1, 6))

            story.append(create_report_table(
                dataframe_to_data(face_df),
                [
                    29*mm, 17*mm, 19*mm, 15*mm,
                    22*mm, 25*mm, 19*mm, 19*mm
                ]
            ))

            # ==================================================
            # 4. FOUR OUTLET BEAMS
            # ==================================================

            story.append(Paragraph(
                "3. FOUR OUTLET BEAM ANALYSIS",
                styles["SectionTitle"]
            ))

            beam_df = pd.DataFrame(outlet_beam_results)

            story.append(create_report_table(
                dataframe_to_data(beam_df),
                [
                    30*mm, 18*mm, 24*mm, 24*mm,
                    25*mm, 23*mm, 26*mm
                ]
            ))

            # ==================================================
            # 5. CORNER COLUMN REACTIONS
            # ==================================================

            story.append(Paragraph(
                "4. CORNER COLUMN REACTIONS",
                styles["SectionTitle"]
            ))

            if corner_column_results:

                column_df = pd.DataFrame(corner_column_results)

                story.append(create_report_table(
                    dataframe_to_data(column_df),
                    [80*mm, 80*mm]
                ))

            else:

                story.append(Paragraph(
                    "Column reaction results are unavailable.",
                    styles["Normal"]
                ))

            story.append(Paragraph(
                "These are axial reactions from the simplified "
                "pinned-beam model, not column capacity checks.",
                styles["ReportNote"]
            ))

            # ==================================================
            # 6. HOPPER SLAB REINFORCEMENT
            # ==================================================

            story.append(Paragraph(
                "5. HOPPER SLAB REINFORCEMENT",
                styles["SectionTitle"]
            ))

            slab = slab_reinforcement_results

            slab_data = [
                ["Parameter", "Calculated value"],
                ["Design moment",
                 f"{slab['Moment']:.3f} kN.m"],
                ["Effective depth",
                 f"{slab['Effective depth']:.2f} mm"],
                ["Main bar diameter",
                 f"{slab['Main diameter']:.0f} mm"],
                ["Main bar spacing",
                 f"{slab['Main spacing']:.2f} mm"],
                ["Main steel required",
                 f"{slab['Main required']:.2f} mm2/m"],
                ["Main steel provided",
                 f"{slab['Main provided']:.2f} mm2/m"],
                ["Distribution bar diameter",
                 f"{slab['Distribution diameter']:.0f} mm"],
                ["Distribution bar spacing",
                 f"{slab['Distribution spacing']:.2f} mm"],
                ["Distribution steel required",
                 f"{slab['Distribution required']:.2f} mm2/m"],
                ["Distribution steel provided",
                 f"{slab['Distribution provided']:.2f} mm2/m"]
            ]

            story.append(create_report_table(
                slab_data,
                [85*mm, 85*mm]
            ))

            # ==================================================
            # 7. BEAM REINFORCEMENT
            # ==================================================

            story.append(Paragraph(
                "6. OUTLET BEAM REINFORCEMENT",
                styles["SectionTitle"]
            ))

            beam_reinf_df = pd.DataFrame(
                beam_reinforcement_results
            )

            story.append(create_report_table(
                dataframe_to_data(beam_reinf_df),
                [
                    29*mm, 23*mm, 23*mm, 13*mm,
                    18*mm, 23*mm, 25*mm, 17*mm, 17*mm
                ]
            ))

            # ==================================================
            # 8. STEP 5 VERIFICATION RESULTS
            # ==================================================

            story.append(Paragraph(
                "7. DESIGN VERIFICATION RESULTS",
                styles["SectionTitle"]
            ))

            verification_df = globals().get(
                "step5_verification_results",
                None
            )

            if verification_df is not None:

                story.append(create_report_table(
                    dataframe_to_data(verification_df),
                    [45*mm, 25*mm, 100*mm]
                ))

            else:

                story.append(Paragraph(
                    "Step 5 results were not saved. Add the following "
                    "two lines after creating results_df in Step 5, "
                    "then rerun Step 5:",
                    styles["ReportNote"]
                ))

                story.append(Paragraph(
                    "step5_verification_results = results_df.copy()",
                    styles["Code"]
                ))

            # ==================================================
            # 9. PRELIMINARY REINFORCEMENT SCHEDULE
            # ==================================================

            story.append(Paragraph(
                "8. PRELIMINARY REINFORCEMENT SCHEDULE",
                styles["SectionTitle"]
            ))

            bbs_data = [
                ["Member", "Reinforcement", "Diameter", "Spacing"],
                [
                    "Hopper slab",
                    "Main steel",
                    f"{slab['Main diameter']:.0f} mm",
                    f"{slab['Main spacing']:.1f} mm c/c"
                ],
                [
                    "Hopper slab",
                    "Distribution steel",
                    f"{slab['Distribution diameter']:.0f} mm",
                    f"{slab['Distribution spacing']:.1f} mm c/c"
                ],
                [
                    "Outlet beams",
                    "Main steel",
                    f"{beam_main_dia.value} mm",
                    "Refer to Step 4"
                ]
            ]

            story.append(create_report_table(
                bbs_data,
                [35*mm, 55*mm, 30*mm, 50*mm]
            ))

            story.append(Paragraph(
                "This is a preliminary reinforcement schedule. "
                "Final bar cutting lengths, hooks, bends, laps "
                "and anchorage details require separate detailing.",
                styles["ReportNote"]
            ))

            # ==================================================
            # 10. DESIGN ASSUMPTIONS
            # ==================================================

            story.append(Paragraph(
                "9. DESIGN ASSUMPTIONS AND LIMITATIONS",
                styles["SectionTitle"]
            ))

            limitations = [
                "1. Four outlet beams are modelled as simply supported "
                "with pinned connections to corner columns.",

                "2. Beam and column reactions follow the simplified "
                "load-transfer model in Step 3.",

                "3. Hopper material pressure is a screening estimate, "
                "not a verified IS 4995 calculation.",

                "4. Slab reinforcement is based on the simplified "
                "strip analysis in Step 4.",

                "5. Step 5 includes only the checks implemented in "
                "that verification module.",

                "6. Independent engineering review, complete load "
                "assessment and reinforcement detailing are required "
                "before construction."
            ]

            for item in limitations:

                story.append(Paragraph(
                    escape(item),
                    styles["ReportNote"]
                ))

                story.append(Spacer(1, 5))

            # ==================================================
            # 11. PAGE NUMBER
            # ==================================================

            def add_page_number(canvas, doc):

                canvas.saveState()

                canvas.setFont("Helvetica", 8)

                canvas.drawString(
                    15*mm,
                    10*mm,
                    "RCC Hopper Bunker Design Report"
                )

                canvas.drawRightString(
                    195*mm,
                    10*mm,
                    f"Page {doc.page}"
                )

                canvas.restoreState()

            # ==================================================
            # 12. GENERATE PDF
            # ==================================================

            doc.build(
                story,
                onFirstPage=add_page_number,
                onLaterPages=add_page_number
            )

            if not os.path.exists(report_filename):
                raise FileNotFoundError(
                    "PDF file was not created."
                )

            print("PDF REPORT GENERATED SUCCESSFULLY")
            print("=" * 50)

            print("File name:")
            print(os.path.basename(report_filename))

            print("\nReport includes:")
            print("1. General design inputs")
            print("2. Hopper face analysis")
            print("3. Four outlet beam analysis")
            print("4. Corner column reactions")
            print("5. Hopper slab reinforcement")
            print("6. Outlet beam reinforcement")
            print("7. Step 5 verification results, if saved")
            print("8. Preliminary reinforcement schedule")
            print("9. Design assumptions and limitations")

            print("\nStarting PDF download...")

            from google.colab import files
            files.download(report_filename)

        except Exception as e:

            print("STEP 6 ERROR:")
            print(str(e))


# ==========================================================
# DISPLAY STEP 6 BUTTON
# ==========================================================

step6_button = widgets.Button(
    description="Generate Final Design Report",
    button_style="success",
    icon="file-pdf"
)

step6_button.on_click(generate_step6_report)

display(widgets.VBox([
    widgets.HTML(
        "<h3>Step 6: Final Design Report and PDF Export</h3>"
    ),
    step6_button,
    step6_output
]))

print("Step 6 is ready. Click Generate Final Design Report.")


ReportLab is ready.


Step 6 is ready. Click Generate Final Design Report.


In [ ]:

# ==========================================================
# STEP 7: RCC REINFORCEMENT DETAILING AND BBS
# WITHOUT PRELIMINARY HOPPER LAYOUT DIAGRAM
# ==========================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
import math

from IPython.display import display, clear_output

step7_output = widgets.Output()

# ==========================================================
# ADDITIONAL DETAILING INPUTS
# ==========================================================

bbs_cover = widgets.FloatText(
    value=40,
    description="Clear cover (mm)",
    style={"description_width": "initial"}
)

slab_hook_length = widgets.FloatText(
    value=150,
    description="Slab anchorage per end (mm)",
    style={"description_width": "initial"}
)

beam_anchorage = widgets.FloatText(
    value=300,
    description="Beam anchorage per end (mm)",
    style={"description_width": "initial"}
)

stirrup_spacing = widgets.FloatText(
    value=150,
    description="Stirrup spacing (mm)",
    style={"description_width": "initial"}
)

stirrup_hook_length = widgets.FloatText(
    value=75,
    description="Stirrup hook extension (mm)",
    style={"description_width": "initial"}
)

lap_length_factor = widgets.FloatText(
    value=50,
    description="Lap length factor (d)",
    style={"description_width": "initial"}
)

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

def steel_unit_weight(diameter):
    return diameter**2 / 162


def number_of_bars(length_mm, spacing_mm):

    if spacing_mm <= 0:
        raise ValueError("Bar spacing must be positive.")

    return max(2, int(math.ceil(length_mm / spacing_mm)) + 1)


def make_bbs_row(
    mark, member, bar_type, diameter,
    quantity, cutting_length
):

    total_length = quantity * cutting_length / 1000

    unit_weight = steel_unit_weight(diameter)

    total_weight = total_length * unit_weight

    return {
        "Bar mark": mark,
        "Member": member,
        "Bar type": bar_type,
        "Diameter (mm)": diameter,
        "Quantity": quantity,
        "Cutting length (mm)": round(cutting_length, 1),
        "Total length (m)": round(total_length, 3),
        "Unit weight (kg/m)": round(unit_weight, 3),
        "Total weight (kg)": round(total_weight, 3)
    }


# ==========================================================
# MAIN BBS GENERATION FUNCTION
# ==========================================================

def generate_step7_bbs(change=None):

    with step7_output:

        clear_output(wait=True)

        try:

            # --------------------------------------------------
            # CHECK PREVIOUS STEPS
            # --------------------------------------------------

            inp = get_inputs()

            if not hopper_face_actions:
                raise ValueError("Run Step 3 first.")

            if not outlet_beam_results:
                raise ValueError("Run Step 3 first.")

            if not beam_reinforcement_results:
                raise ValueError("Run Step 4 first.")

            if not isinstance(slab_reinforcement_results, dict):
                raise ValueError("Run Step 4 first.")

            # --------------------------------------------------
            # READ DIMENSIONS
            # --------------------------------------------------

            L = float(inp["L_outer"])
            W = float(inp["W_outer"])
            t = float(inp["t_wall"])

            Hh = float(inp["H_hopper"])

            Lo = float(inp["L_outlet"])
            Wo = float(inp["W_outlet"])

            Li = L - 2*t
            Wi = W - 2*t

            cover_mm = float(bbs_cover.value)
            slab_anchor = float(slab_hook_length.value)
            beam_anchor = float(beam_anchorage.value)
            stirrup_s = float(stirrup_spacing.value)
            stirrup_hook = float(stirrup_hook_length.value)

            if min(Li, Wi, Lo, Wo, Hh) <= 0:
                raise ValueError("Invalid bunker dimensions.")

            if Lo >= Li or Wo >= Wi:
                raise ValueError(
                    "Outlet dimensions must be smaller than internal dimensions."
                )

            if min(cover_mm, stirrup_s, stirrup_hook) <= 0:
                raise ValueError(
                    "Cover, stirrup spacing and hook length must be positive."
                )

            if min(slab_anchor, beam_anchor) < 0:
                raise ValueError("Anchorage lengths cannot be negative.")

            # --------------------------------------------------
            # REINFORCEMENT INPUTS FROM STEP 4
            # --------------------------------------------------

            slab = slab_reinforcement_results

            main_dia = float(slab["Main diameter"])
            dist_dia = float(slab["Distribution diameter"])

            main_spacing = float(slab["Main spacing"])
            dist_spacing = float(slab["Distribution spacing"])

            beam_dia = float(beam_main_dia.value)
            stirrup_dia_value = float(stirrup_dia.value)

            beam_b = float(outlet_beam_width.value)
            beam_D = float(outlet_beam_depth.value)

            if min(
                main_dia, dist_dia,
                main_spacing, dist_spacing,
                beam_dia, stirrup_dia_value,
                beam_b, beam_D
            ) <= 0:
                raise ValueError(
                    "Check reinforcement diameters, spacing and beam dimensions."
                )

            # --------------------------------------------------
            # CREATE BBS
            # --------------------------------------------------

            bbs_rows = []
            slab_detail_rows = []
            beam_detail_rows = []

            mark = 1

            # ==================================================
            # A. HOPPER SLAB REINFORCEMENT
            # ==================================================

            faces = [
                {
                    "name": "Longitudinal face 1",
                    "run": (Wi-Wo)/2,
                    "edge": Lo
                },
                {
                    "name": "Longitudinal face 2",
                    "run": (Wi-Wo)/2,
                    "edge": Lo
                },
                {
                    "name": "Transverse face 1",
                    "run": (Li-Lo)/2,
                    "edge": Wo
                },
                {
                    "name": "Transverse face 2",
                    "run": (Li-Lo)/2,
                    "edge": Wo
                }
            ]

            for face in faces:

                run = face["run"]
                edge = face["edge"]

                slope_length = math.sqrt(run**2 + Hh**2)

                slope_length_mm = slope_length*1000
                edge_mm = edge*1000

                # Main bars along slope

                main_count = number_of_bars(
                    edge_mm-2*cover_mm,
                    main_spacing
                )

                main_cutting = (
                    slope_length_mm
                    - 2*cover_mm
                    + 2*slab_anchor
                )

                if main_cutting <= 0:
                    raise ValueError(
                        "Invalid slab main bar cutting length."
                    )

                bbs_rows.append(
                    make_bbs_row(
                        mark,
                        face["name"],
                        "Slab main bars",
                        main_dia,
                        main_count,
                        main_cutting
                    )
                )

                slab_detail_rows.append({
                    "Face": face["name"],
                    "Direction": "Along slope",
                    "Bars": main_count,
                    "Diameter (mm)": main_dia,
                    "Spacing (mm)": main_spacing,
                    "Slope length (m)": slope_length,
                    "Cutting length (mm)": main_cutting
                })

                mark += 1

                # Distribution bars across face

                dist_count = number_of_bars(
                    slope_length_mm-2*cover_mm,
                    dist_spacing
                )

                dist_cutting = (
                    edge_mm
                    - 2*cover_mm
                    + 2*slab_anchor
                )

                if dist_cutting <= 0:
                    raise ValueError(
                        "Invalid slab distribution bar cutting length."
                    )

                bbs_rows.append(
                    make_bbs_row(
                        mark,
                        face["name"],
                        "Slab distribution bars",
                        dist_dia,
                        dist_count,
                        dist_cutting
                    )
                )

                slab_detail_rows.append({
                    "Face": face["name"],
                    "Direction": "Across face",
                    "Bars": dist_count,
                    "Diameter (mm)": dist_dia,
                    "Spacing (mm)": dist_spacing,
                    "Slope length (m)": slope_length,
                    "Cutting length (mm)": dist_cutting
                })

                mark += 1

            # ==================================================
            # B. FOUR OUTLET BEAMS - MAIN REINFORCEMENT
            # ==================================================

            for beam in outlet_beam_results:

                beam_name = beam["Beam"]
                span = float(beam["Span (m)"])

                matching = [
                    x for x in beam_reinforcement_results
                    if x["Beam"] == beam_name
                ]

                if not matching:
                    raise ValueError(
                        "Reinforcement result missing for " + beam_name
                    )

                reinforcement = matching[0]

                bar_count = int(reinforcement["Bars"])

                beam_cutting = (
                    span*1000
                    - 2*cover_mm
                    + 2*beam_anchor
                )

                if beam_cutting <= 0:
                    raise ValueError(
                        "Invalid beam bar cutting length."
                    )

                bbs_rows.append(
                    make_bbs_row(
                        mark,
                        beam_name,
                        "Beam main bars - preliminary",
                        beam_dia,
                        bar_count,
                        beam_cutting
                    )
                )

                beam_detail_rows.append({
                    "Beam": beam_name,
                    "Main bars": bar_count,
                    "Main diameter (mm)": beam_dia,
                    "Cutting length (mm)": beam_cutting,
                    "Span (m)": span
                })

                mark += 1

                # ==================================================
                # C. BEAM STIRRUPS
                # ==================================================

                clear_length_mm = span*1000-2*cover_mm

                stirrup_count = (
                    int(math.ceil(clear_length_mm/stirrup_s))+1
                )

                stirrup_width = (
                    beam_b-2*cover_mm/1000*1000-stirrup_dia_value
                )

                stirrup_depth = (
                    beam_D-2*cover_mm/1000*1000-stirrup_dia_value
                )

                if min(stirrup_width, stirrup_depth) <= 0:
                    raise ValueError(
                        "Beam section is too small for the selected cover and stirrup."
                    )

                stirrup_cutting = (
                    2*(stirrup_width+stirrup_depth)
                    + 2*stirrup_hook
                )

                bbs_rows.append(
                    make_bbs_row(
                        mark,
                        beam_name,
                        "Beam rectangular stirrups - preliminary",
                        stirrup_dia_value,
                        stirrup_count,
                        stirrup_cutting
                    )
                )

                mark += 1

            # ==================================================
            # D. FINAL DATAFRAME
            # ==================================================

            bbs_df = pd.DataFrame(bbs_rows)

            total_weight = bbs_df["Total weight (kg)"].sum()
            total_length = bbs_df["Total length (m)"].sum()
            total_bars = bbs_df["Quantity"].sum()

            global step7_bbs_results
            global step7_slab_details
            global step7_beam_details

            step7_bbs_results = bbs_df.copy()

            step7_slab_details = pd.DataFrame(slab_detail_rows)
            step7_beam_details = pd.DataFrame(beam_detail_rows)

            # ==================================================
            # E. DISPLAY RESULTS
            # ==================================================

            print("="*65)
            print("STEP 7: RCC REINFORCEMENT DETAILING AND BBS")
            print("="*65)

            print("\nA. HOPPER SLAB REINFORCEMENT DETAILING")

            display(step7_slab_details.round(2))

            print("\nB. OUTLET BEAM REINFORCEMENT DETAILING")

            display(step7_beam_details.round(2))

            print("\nC. COMPLETE BAR BENDING SCHEDULE")

            display(bbs_df.round(2))

            print("\nD. TOTAL REINFORCEMENT SUMMARY")

            print(f"Total number of bars = {total_bars}")
            print(f"Total steel length = {total_length:.2f} m")
            print(f"Estimated steel weight = {total_weight:.2f} kg")

            # ==================================================
            # F. EXPORT EXCEL FILE
            # ==================================================

            excel_filename = "/content/RCC_Hopper_Bunker_BBS.xlsx"

            with pd.ExcelWriter(excel_filename) as writer:

                bbs_df.to_excel(
                    writer,
                    sheet_name="Bar Bending Schedule",
                    index=False
                )

                step7_slab_details.to_excel(
                    writer,
                    sheet_name="Slab Detailing",
                    index=False
                )

                step7_beam_details.to_excel(
                    writer,
                    sheet_name="Beam Detailing",
                    index=False
                )

                summary_df = pd.DataFrame({
                    "Description": [
                        "Total number of bars",
                        "Total steel length (m)",
                        "Estimated steel weight (kg)"
                    ],
                    "Value": [
                        total_bars,
                        total_length,
                        total_weight
                    ]
                })

                summary_df.to_excel(
                    writer,
                    sheet_name="Summary",
                    index=False
                )

            print("\nEXCEL BBS FILE CREATED SUCCESSFULLY")

            from google.colab import files

            print("Starting BBS Excel download...")
            files.download(excel_filename)

        except Exception as e:

            print("\nSTEP 7 ERROR:")
            print(str(e))


# ==========================================================
# DISPLAY STEP 7
# ==========================================================

step7_button = widgets.Button(
    description="Generate Reinforcement Details and BBS",
    button_style="success",
    icon="calculator"
)

step7_button.on_click(generate_step7_bbs)

display(widgets.VBox([
    widgets.HTML(
        "<h3>Step 7: RCC Reinforcement Detailing and BBS</h3>"
    ),
    bbs_cover,
    slab_hook_length,
    beam_anchorage,
    stirrup_spacing,
    stirrup_hook_length,
    lap_length_factor,
    step7_button,
    step7_output
]))

print("Step 7 is ready. Run Steps 3 and 4, then click the button.")


Step 7 is ready. Run Steps 3 and 4, then click the button.


In [ ]:

# ================================================================
# STEP 8: COMPLETE BUNKER REINFORCEMENT DETAILING
# 2D PLAN + 2D SECTION + 3D REINFORCEMENT MODEL
#
# Arrangement:
# 1. Four corner columns
# 2. Four horizontal edge beams at the top
# 3. Vertical side walls between columns
# 4. Four sloping hopper slabs
# 5. Four outlet supporting beams
# ================================================================

import math
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, clear_output

# ------------------------------------------------
# 1. GET ORIGINAL INPUTS FROM PREVIOUS CELLS
# ------------------------------------------------

inp8 = get_inputs()

L = float(inp8["L_outer"])
W = float(inp8["W_outer"])
tw = float(inp8["t_wall"])
Hw = float(inp8["H_wall"])
Hh = float(inp8["H_hopper"])
Lo = float(inp8["L_outlet"])
Wo = float(inp8["W_outlet"])

# Interior dimensions
Li = L - 2 * tw
Wi = W - 2 * tw

if min(Li, Wi, Lo, Wo, Hw, Hh) <= 0:
    raise ValueError("Check bunker dimensions. All internal dimensions and heights must be positive.")

if Lo >= Li or Wo >= Wi:
    raise ValueError("Outlet dimensions must be smaller than the internal bunker dimensions.")

# Original structural dimensions from previous cells, if available
def get_widget_value(name, default):
    obj = globals().get(name, None)
    if obj is not None and hasattr(obj, "value"):
        return float(obj.value)
    return float(default)

beam_b = get_widget_value("outlet_beam_width", 300)
beam_D = get_widget_value("outlet_beam_depth", 450)

col_b = get_widget_value("column_width", 300)
col_D = get_widget_value("column_depth", 300)

col_height = get_widget_value("column_height", 3.0)

# ------------------------------------------------
# 2. STEP 8 REINFORCEMENT INPUTS
# ------------------------------------------------

wall_vertical_dia = widgets.IntText(
    value=16, description="Wall vertical dia (mm)"
)

wall_vertical_spacing = widgets.IntText(
    value=150, description="Wall vertical spacing (mm)"
)

wall_horizontal_dia = widgets.IntText(
    value=12, description="Wall horizontal dia (mm)"
)

wall_horizontal_spacing = widgets.IntText(
    value=200, description="Wall horizontal spacing (mm)"
)

column_bar_dia = widgets.IntText(
    value=20, description="Column main dia (mm)"
)

column_bar_count = widgets.IntText(
    value=8, description="Column main bars"
)

column_tie_dia = widgets.IntText(
    value=8, description="Column tie dia (mm)"
)

column_tie_spacing = widgets.IntText(
    value=150, description="Column tie spacing (mm)"
)

edge_beam_bar_dia = widgets.IntText(
    value=16, description="Edge beam bar dia (mm)"
)

edge_beam_stirrup_dia = widgets.IntText(
    value=8, description="Edge beam stirrup dia (mm)"
)

beam_stirrup_spacing_8 = widgets.IntText(
    value=150, description="Beam stirrup spacing (mm)"
)

hopper_main_dia_8 = widgets.IntText(
    value=12, description="Hopper main dia (mm)"
)

hopper_main_spacing_8 = widgets.IntText(
    value=150, description="Hopper main spacing (mm)"
)

hopper_distribution_dia_8 = widgets.IntText(
    value=10, description="Hopper distribution dia (mm)"
)

hopper_distribution_spacing_8 = widgets.IntText(
    value=150, description="Hopper distribution spacing (mm)"
)

outlet_beam_bar_dia_8 = widgets.IntText(
    value=20, description="Outlet beam bar dia (mm)"
)

outlet_beam_stirrup_dia_8 = widgets.IntText(
    value=8, description="Outlet beam stirrup dia (mm)"
)

cover_8 = widgets.IntText(
    value=int(inp8["cover_mm"]), description="Concrete cover (mm)"
)

reinforcement_controls = widgets.VBox([
    widgets.HTML("<h3>Wall reinforcement</h3>"),
    wall_vertical_dia,
    wall_vertical_spacing,
    wall_horizontal_dia,
    wall_horizontal_spacing,

    widgets.HTML("<h3>Corner column reinforcement</h3>"),
    column_bar_dia,
    column_bar_count,
    column_tie_dia,
    column_tie_spacing,

    widgets.HTML("<h3>Top horizontal edge beam reinforcement</h3>"),
    edge_beam_bar_dia,
    edge_beam_stirrup_dia,

    widgets.HTML("<h3>Hopper slab reinforcement</h3>"),
    hopper_main_dia_8,
    hopper_main_spacing_8,
    hopper_distribution_dia_8,
    hopper_distribution_spacing_8,

    widgets.HTML("<h3>Bottom outlet beam reinforcement</h3>"),
    outlet_beam_bar_dia_8,
    outlet_beam_stirrup_dia_8,

    cover_8
])

display(reinforcement_controls)

# ------------------------------------------------
# 3. GEOMETRY
# ------------------------------------------------

# Centre coordinates
x1, x2 = -L/2, L/2
y1, y2 = -W/2, W/2

# Outlet coordinates
ox1, ox2 = -Lo/2, Lo/2
oy1, oy2 = -Wo/2, Wo/2

# Hopper top level
z_top_hopper = 0.0

# Outlet bottom level
z_outlet = -Hh

# Column bottom level
z_column_bottom = z_outlet - col_height

# Concrete cover in metres
cover_m = cover_8.value / 1000

# ------------------------------------------------
# 4. VALIDATION OF REINFORCEMENT INPUTS
# ------------------------------------------------

def validate_reinforcement_inputs():

    widgets_to_check = [
        wall_vertical_dia,
        wall_vertical_spacing,
        wall_horizontal_dia,
        wall_horizontal_spacing,
        column_bar_dia,
        column_bar_count,
        column_tie_dia,
        column_tie_spacing,
        edge_beam_bar_dia,
        edge_beam_stirrup_dia,
        beam_stirrup_spacing_8,
        hopper_main_dia_8,
        hopper_main_spacing_8,
        hopper_distribution_dia_8,
        hopper_distribution_spacing_8,
        outlet_beam_bar_dia_8,
        outlet_beam_stirrup_dia_8,
        cover_8
    ]

    for widget in widgets_to_check:
        if widget.value <= 0:
            raise ValueError(
                f"{widget.description}: value must be greater than zero."
            )

    if column_bar_count.value < 4:
        raise ValueError("Column must have at least 4 longitudinal bars.")

validate_reinforcement_inputs()


# ================================================================
# CORRECTED 2D PLAN VIEW
# Main bars: run down each sloping face
# Distribution bars: parallel to perimeter and outlet edges
# Hip bars: along four hopper corners
# ================================================================

def draw_plan_view():

    fig, ax = plt.subplots(figsize=(12, 10))

    # Four top perimeter corners (inside wall)
    P = [
        (-Li/2, -Wi/2),
        ( Li/2, -Wi/2),
        ( Li/2,  Wi/2),
        (-Li/2,  Wi/2)
    ]

    # Four outlet corners
    Q = [
        (-Lo/2, -Wo/2),
        ( Lo/2, -Wo/2),
        ( Lo/2,  Wo/2),
        (-Lo/2,  Wo/2)
    ]

    # ------------------------------------------------
    # OUTER WALL AND INNER WALL
    # ------------------------------------------------

    outer_x = [-L/2, L/2, L/2, -L/2, -L/2]
    outer_y = [-W/2, -W/2, W/2, W/2, -W/2]

    inner_x = [-Li/2, Li/2, Li/2, -Li/2, -Li/2]
    inner_y = [-Wi/2, -Wi/2, Wi/2, Wi/2, -Wi/2]

    ax.plot(
        outer_x, outer_y,
        color="black", linewidth=3,
        label="Outer bunker wall"
    )

    ax.plot(
        inner_x, inner_y,
        color="gray", linewidth=2,
        linestyle="--",
        label="Inner wall face"
    )

    # ------------------------------------------------
    # TOP EDGE BEAMS
    # ------------------------------------------------

    for i in range(4):

        j = (i+1) % 4

        ax.plot(
            [P[i][0], P[j][0]],
            [P[i][1], P[j][1]],
            color="purple",
            linewidth=6,
            label="Top edge beams" if i == 0 else None
        )

    # ------------------------------------------------
    # CENTRAL OUTLET BEAMS AND OPENING
    # ------------------------------------------------

    for i in range(4):

        j = (i+1) % 4

        ax.plot(
            [Q[i][0], Q[j][0]],
            [Q[i][1], Q[j][1]],
            color="darkorange",
            linewidth=5,
            label="Outlet supporting beams" if i == 0 else None
        )

    # Outlet opening
    outlet_x = [Q[0][0], Q[1][0], Q[2][0], Q[3][0], Q[0][0]]
    outlet_y = [Q[0][1], Q[1][1], Q[2][1], Q[3][1], Q[0][1]]

    ax.fill(
        outlet_x, outlet_y,
        color="white",
        zorder=3
    )

    ax.plot(
        outlet_x, outlet_y,
        color="black",
        linewidth=2,
        zorder=4
    )

    # ------------------------------------------------
    # HOPPER MAIN REINFORCEMENT
    # Blue bars run from the top edge to outlet edge
    # ------------------------------------------------

    main_positions = np.linspace(0.06, 0.94, 13)

    for i in range(4):

        j = (i+1) % 4

        for u in main_positions:

            # Position along top edge
            start_x = P[i][0] + u*(P[j][0] - P[i][0])
            start_y = P[i][1] + u*(P[j][1] - P[i][1])

            # Corresponding position along outlet edge
            end_x = Q[i][0] + u*(Q[j][0] - Q[i][0])
            end_y = Q[i][1] + u*(Q[j][1] - Q[i][1])

            ax.plot(
                [start_x, end_x],
                [start_y, end_y],
                color="blue",
                linewidth=1.3,
                zorder=2,
                label="Hopper main bars" if i == 0 and u == main_positions[0] else None
            )

    # ------------------------------------------------
    # HOPPER DISTRIBUTION REINFORCEMENT
    # Green bars run parallel to the top and outlet edges
    # ------------------------------------------------

    distribution_positions = np.linspace(0.10, 0.90, 9)

    for i in range(4):

        j = (i+1) % 4

        for v in distribution_positions:

            # Point on first sloping edge
            start_x = P[i][0] + v*(Q[i][0] - P[i][0])
            start_y = P[i][1] + v*(Q[i][1] - P[i][1])

            # Point on second sloping edge
            end_x = P[j][0] + v*(Q[j][0] - P[j][0])
            end_y = P[j][1] + v*(Q[j][1] - P[j][1])

            ax.plot(
                [start_x, end_x],
                [start_y, end_y],
                color="green",
                linewidth=1.3,
                linestyle="--",
                zorder=2,
                label="Hopper distribution bars"
                if i == 0 and v == distribution_positions[0] else None
            )

    # ------------------------------------------------
    # FOUR HIP / CORNER BARS
    # ------------------------------------------------

    for i in range(4):

        ax.plot(
            [P[i][0], Q[i][0]],
            [P[i][1], Q[i][1]],
            color="black",
            linewidth=2.5,
            linestyle=":",
            label="Hopper hip bars" if i == 0 else None
        )

    # ------------------------------------------------
    # FOUR CORNER COLUMNS WITH LONGITUDINAL BARS
    # ------------------------------------------------

    column_positions = [
        (x1, y1),
        (x2, y1),
        (x2, y2),
        (x1, y2)
    ]

    column_labels = ["C1", "C2", "C3", "C4"]

    for index, (cx, cy) in enumerate(column_positions):

        ax.add_patch(
            plt.Rectangle(
                (cx-col_b/2000, cy-col_D/2000),
                col_b/1000,
                col_D/1000,
                fill=False,
                edgecolor="black",
                linewidth=2.5,
                zorder=5
            )
        )

        # Main column bars
        n = column_bar_count.value

        rx = col_b/2000 - cover_m
        ry = col_D/2000 - cover_m

        for k in range(n):

            angle = 2*np.pi*k/n

            bx = cx + rx*np.cos(angle)
            by = cy + ry*np.sin(angle)

            ax.plot(
                bx, by,
                marker="o",
                color="red",
                markersize=4,
                zorder=6
            )

        # Column labels outside the plan
        label_offsets = [
            (-0.35, -0.35),
            (0.35, -0.35),
            (0.35, 0.35),
            (-0.35, 0.35)
        ]

        dx, dy = label_offsets[index]

        ax.text(
            cx+dx,
            cy+dy,
            column_labels[index],
            fontsize=11,
            fontweight="bold",
            ha="center",
            va="center",
            bbox=dict(facecolor="white", edgecolor="black", alpha=0.9)
        )

    # ------------------------------------------------
    # WALL VERTICAL REINFORCEMENT (PLAN REPRESENTATION)
    # ------------------------------------------------

    spacing = wall_vertical_spacing.value/1000

    # Front and rear walls
    for xx in np.arange(-Li/2, Li/2+0.001, spacing):

        for yy in [-W/2+cover_m, W/2-cover_m]:

            ax.plot(
                xx, yy,
                marker="o",
                color="red",
                markersize=2.8
            )

    # Left and right walls
    for yy in np.arange(-Wi/2, Wi/2+0.001, spacing):

        for xx in [-L/2+cover_m, L/2-cover_m]:

            ax.plot(
                xx, yy,
                marker="o",
                color="red",
                markersize=2.8
            )

    # ------------------------------------------------
    # CENTRAL LABEL
    # ------------------------------------------------

    ax.text(
        0, 0,
        "HOPPER\nOPENING",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        zorder=7
    )

    ax.set_title(
        "BUNKER REINFORCEMENT - 2D PLAN",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Length (m)")
    ax.set_ylabel("Width (m)")

    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)

    # Keep legend outside the drawing
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        fontsize=9
    )

    plt.tight_layout()
    plt.show()

# ------------------------------------------------
# 6. 2D SECTION / ELEVATION WITH REINFORCEMENT
# ------------------------------------------------

def draw_section_view():

    fig, ax = plt.subplots(figsize=(12, 9))

    # Section through length
    # Side walls
    ax.plot([x1, x1], [0, Hw], color="black", linewidth=5)
    ax.plot([x2, x2], [0, Hw], color="black", linewidth=5)

    # Top edge beam
    ax.plot([x1, x2], [Hw, Hw], color="purple", linewidth=7)

    # Hopper slab
    ax.plot([x1+tw, ox1], [0, z_outlet],
            color="black", linewidth=5)

    ax.plot([x2-tw, ox2], [0, z_outlet],
            color="black", linewidth=5)

    # Outlet beam
    ax.plot([ox1, ox2], [z_outlet, z_outlet],
            color="orange", linewidth=7)

    # Four corner columns (two visible in section)
    for cx in [x1, x2]:

        ax.plot(
            [cx, cx],
            [z_column_bottom, 0],
            color="black",
            linewidth=7
        )

        # Column main reinforcement
        for offset in [-col_b/2000, col_b/2000]:
            ax.plot(
                [cx+offset, cx+offset],
                [z_column_bottom, 0],
                color="red",
                linewidth=1.5
            )

        # Column ties
        for zz in np.arange(
            z_column_bottom,
            0,
            column_tie_spacing.value/1000
        ):
            ax.plot(
                [cx-col_b/2000, cx+col_b/2000],
                [zz, zz],
                color="green",
                linewidth=1
            )

    # Wall vertical reinforcement
    v_spacing = wall_vertical_spacing.value/1000

    for xx in np.arange(x1+tw, x2-tw, v_spacing):
        ax.plot(
            [xx, xx],
            [0, Hw],
            color="red",
            linewidth=1
        )

    # Wall horizontal reinforcement
    h_spacing = wall_horizontal_spacing.value/1000

    for zz in np.arange(0, Hw, h_spacing):
        ax.plot(
            [x1+tw, x2-tw],
            [zz, zz],
            color="blue",
            linewidth=1
        )

    # Hopper reinforcement bars
    # Two layers are represented on each sloping face
    for f in np.linspace(0.08, 0.92, 10):

        xx_left = (x1+tw) + f*(ox1-(x1+tw))
        zz_left = 0 + f*z_outlet

        xx_right = (x2-tw) + f*(ox2-(x2-tw))
        zz_right = 0 + f*z_outlet

        ax.plot(
            [xx_left, xx_left],
            [zz_left-0.04, zz_left+0.04],
            color="blue",
            linewidth=2
        )

        ax.plot(
            [xx_right, xx_right],
            [zz_right-0.04, zz_right+0.04],
            color="blue",
            linewidth=2
        )

    # Labels
    ax.text(0, Hw+0.2, "TOP EDGE BEAM",
            ha="center", color="purple", fontweight="bold")

    ax.text(0, z_outlet-0.35, "BOTTOM OUTLET BEAM",
            ha="center", color="darkorange", fontweight="bold")

    ax.text(x1-0.25, Hw/2, "SIDE WALL",
            rotation=90, ha="center")

    ax.text(x1-0.35, z_column_bottom/2,
            "CORNER COLUMN", rotation=90, ha="center")

    ax.text(0, -Hh/2, "SLOPING HOPPER SLAB",
            ha="center", fontweight="bold")

    ax.set_title(
        "BUNKER REINFORCEMENT - 2D SECTION",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Length (m)")
    ax.set_ylabel("Height (m)")
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal", adjustable="datalim")

    plt.show()


# ------------------------------------------------
# 7. 3D REINFORCEMENT MODEL
# ------------------------------------------------

def draw_3d_reinforcement():

    fig = go.Figure()

    def add_line(points, color="black", width=5, name=None):
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        zs = [p[2] for p in points]

        fig.add_trace(
            go.Scatter3d(
                x=xs, y=ys, z=zs,
                mode="lines",
                line=dict(color=color, width=width),
                name=name,
                showlegend=(name is not None)
            )
        )

    # --------------------------------------------
    # CONCRETE WALL EDGES
    # --------------------------------------------

    corners = [
        (x1, y1),
        (x2, y1),
        (x2, y2),
        (x1, y2)
    ]

    for i in range(4):

        a = corners[i]
        b = corners[(i+1) % 4]

        add_line(
            [(a[0], a[1], 0),
             (a[0], a[1], Hw),
             (b[0], b[1], Hw),
             (b[0], b[1], 0)],
            color="gray",
            width=7,
            name="Bunker wall outline" if i == 0 else None
        )

    # --------------------------------------------
    # TOP HORIZONTAL EDGE BEAMS
    # --------------------------------------------

    top_beam_lines = [
        [(x1, y1, Hw), (x2, y1, Hw)],
        [(x2, y1, Hw), (x2, y2, Hw)],
        [(x2, y2, Hw), (x1, y2, Hw)],
        [(x1, y2, Hw), (x1, y1, Hw)]
    ]

    for i, line in enumerate(top_beam_lines):
        add_line(
            line,
            color="purple",
            width=10,
            name="Top edge beams" if i == 0 else None
        )

    # --------------------------------------------
    # FOUR CORNER COLUMNS
    # --------------------------------------------

    column_positions = [
        (x1, y1),
        (x2, y1),
        (x2, y2),
        (x1, y2)
    ]

    for i, (cx, cy) in enumerate(column_positions):

        # Concrete column outline
        add_line(
            [
                (cx-col_b/2000, cy-col_D/2000, z_column_bottom),
                (cx+col_b/2000, cy-col_D/2000, z_column_bottom),
                (cx+col_b/2000, cy+col_D/2000, z_column_bottom),
                (cx-col_b/2000, cy+col_D/2000, z_column_bottom),
                (cx-col_b/2000, cy-col_D/2000, z_column_bottom)
            ],
            color="gray",
            width=5,
            name="Corner columns" if i == 0 else None
        )

        # Longitudinal column bars
        n = column_bar_count.value
        rx = col_b/2000 - cover_m
        ry = col_D/2000 - cover_m

        for j in range(n):

            angle = 2*np.pi*j/n
            bx = cx + rx*np.cos(angle)
            by = cy + ry*np.sin(angle)

            add_line(
                [(bx, by, z_column_bottom),
                 (bx, by, 0)],
                color="red",
                width=5,
                name="Column main bars" if i == 0 and j == 0 else None
            )

        # Column ties
        tie_spacing = column_tie_spacing.value/1000

        for zz in np.arange(
            z_column_bottom,
            0,
            tie_spacing
        ):

            add_line(
                [
                    (cx-rx, cy-ry, zz),
                    (cx+rx, cy-ry, zz),
                    (cx+rx, cy+ry, zz),
                    (cx-rx, cy+ry, zz),
                    (cx-rx, cy-ry, zz)
                ],
                color="green",
                width=3,
                name="Column ties" if i == 0 and zz == z_column_bottom else None
            )

    # --------------------------------------------
    # WALL VERTICAL REINFORCEMENT
    # --------------------------------------------

    v_spacing = wall_vertical_spacing.value/1000

    # Front and rear walls
    for xx in np.arange(x1+tw, x2-tw+0.001, v_spacing):

        for yy in [y1+cover_m, y2-cover_m]:

            add_line(
                [(xx, yy, 0), (xx, yy, Hw)],
                color="red",
                width=3,
                name="Wall vertical bars" if xx == x1+tw else None
            )

    # Left and right walls
    for yy in np.arange(y1+tw, y2-tw+0.001, v_spacing):

        for xx in [x1+cover_m, x2-cover_m]:

            add_line(
                [(xx, yy, 0), (xx, yy, Hw)],
                color="red",
                width=3
            )

    # --------------------------------------------
    # WALL HORIZONTAL REINFORCEMENT
    # --------------------------------------------

    h_spacing = wall_horizontal_spacing.value/1000

    for zz in np.arange(0, Hw, h_spacing):

        add_line(
            [(x1+tw, y1+cover_m, zz),
             (x2-tw, y1+cover_m, zz)],
            color="blue",
            width=3,
            name="Wall horizontal bars" if zz == 0 else None
        )

        add_line(
            [(x1+tw, y2-cover_m, zz),
             (x2-tw, y2-cover_m, zz)],
            color="blue",
            width=3
        )

        add_line(
            [(x1+cover_m, y1+tw, zz),
             (x1+cover_m, y2-tw, zz)],
            color="blue",
            width=3
        )

        add_line(
            [(x2-cover_m, y1+tw, zz),
             (x2-cover_m, y2-tw, zz)],
            color="blue",
            width=3
        )

    # --------------------------------------------
    # FOUR HOPPER CONCRETE FACES
    # --------------------------------------------

    outer_points = [
        (x1+tw, y1+tw, 0),
        (x2-tw, y1+tw, 0),
        (x2-tw, y2-tw, 0),
        (x1+tw, y2-tw, 0)
    ]

    outlet_points = [
        (ox1, oy1, z_outlet),
        (ox2, oy1, z_outlet),
        (ox2, oy2, z_outlet),
        (ox1, oy2, z_outlet)
    ]

    for i in range(4):

        j = (i+1) % 4

        p1 = outer_points[i]
        p2 = outer_points[j]
        p3 = outlet_points[j]
        p4 = outlet_points[i]

        fig.add_trace(
            go.Mesh3d(
                x=[p1[0], p2[0], p3[0], p4[0]],
                y=[p1[1], p2[1], p3[1], p4[1]],
                z=[p1[2], p2[2], p3[2], p4[2]],
                i=[0, 0],
                j=[1, 2],
                k=[2, 3],
                opacity=0.15,
                color="lightgray",
                name="Hopper slab",
                showlegend=(i == 0)
            )
        )

    # --------------------------------------------
    # HOPPER MAIN REINFORCEMENT
    # --------------------------------------------

    # Bars on each sloping face, interpolated between
    # the top perimeter and outlet perimeter.

    n_hopper = max(
        2,
        int((Li*1000)/hopper_main_spacing_8.value)
    )

    for i in range(4):

        j = (i+1) % 4

        p1 = np.array(outer_points[i], dtype=float)
        p2 = np.array(outer_points[j], dtype=float)

        q1 = np.array(outlet_points[i], dtype=float)
        q2 = np.array(outlet_points[j], dtype=float)

        for f in np.linspace(0.05, 0.95, n_hopper):

            start = p1 + f*(p2-p1)
            end = q1 + f*(q2-q1)

            add_line(
                [tuple(start), tuple(end)],
                color="blue",
                width=4,
                name="Hopper main bars" if i == 0 and f < 0.06 else None
            )

    # --------------------------------------------
    # HOPPER DISTRIBUTION REINFORCEMENT
    # --------------------------------------------

    for i in range(4):

        j = (i+1) % 4

        p1 = np.array(outer_points[i], dtype=float)
        p2 = np.array(outer_points[j], dtype=float)

        q1 = np.array(outlet_points[i], dtype=float)
        q2 = np.array(outlet_points[j], dtype=float)

        for f in np.linspace(0.1, 0.9, 9):

            start = p1 + f*(q1-p1)
            end = p2 + f*(q2-p2)

            add_line(
                [tuple(start), tuple(end)],
                color="orange",
                width=3,
                name="Hopper distribution bars"
                if i == 0 and f < 0.12 else None
            )

    # --------------------------------------------
    # FOUR OUTLET SUPPORTING BEAMS
    # --------------------------------------------

    outlet_beams = [
        [(ox1, oy1, z_outlet), (ox2, oy1, z_outlet)],
        [(ox2, oy1, z_outlet), (ox2, oy2, z_outlet)],
        [(ox2, oy2, z_outlet), (ox1, oy2, z_outlet)],
        [(ox1, oy2, z_outlet), (ox1, oy1, z_outlet)]
    ]

    for i, line in enumerate(outlet_beams):

        add_line(
            line,
            color="darkorange",
            width=10,
            name="Outlet supporting beams" if i == 0 else None
        )

    # --------------------------------------------
    # OUTLET OPENING
    # --------------------------------------------

    add_line(
        [
            (ox1, oy1, z_outlet),
            (ox2, oy1, z_outlet),
            (ox2, oy2, z_outlet),
            (ox1, oy2, z_outlet),
            (ox1, oy1, z_outlet)
        ],
        color="black",
        width=5,
        name="Outlet opening"
    )

    # --------------------------------------------
    # DISPLAY 3D MODEL
    # --------------------------------------------

    fig.update_layout(
        title="3D RCC BUNKER - COMPLETE REINFORCEMENT DETAILING",
        scene=dict(
            xaxis_title="Length (m)",
            yaxis_title="Width (m)",
            zaxis_title="Height (m)",
            aspectmode="data"
        ),
        height=1000,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.15,
            xanchor="left",
            x=0
        )
    )

    fig.show()


# ------------------------------------------------
# 8. REINFORCEMENT DETAILING SUMMARY
# ------------------------------------------------

def show_reinforcement_summary():

    print("\n========== BUNKER REINFORCEMENT DETAILS ==========")

    print("\n1. CORNER COLUMNS")
    print("Number of columns: 4")
    print("Column size:", col_b, "mm x", col_D, "mm")
    print("Main bars:", column_bar_count.value,
          "nos of", column_bar_dia.value, "mm")
    print("Column ties:", column_tie_dia.value, "mm dia")
    print("Tie spacing:", column_tie_spacing.value, "mm")

    print("\n2. TOP EDGE BEAMS")
    print("Four horizontal edge beams")
    print("Beam size:", beam_b, "mm x", beam_D, "mm")
    print("Longitudinal bar dia:", edge_beam_bar_dia.value, "mm")
    print("Stirrup dia:", edge_beam_stirrup_dia.value, "mm")

    print("\n3. VERTICAL SIDE WALLS")
    print("Wall thickness:", tw, "m")
    print("Vertical bars:", wall_vertical_dia.value, "mm")
    print("Vertical spacing:", wall_vertical_spacing.value, "mm")
    print("Horizontal bars:", wall_horizontal_dia.value, "mm")
    print("Horizontal spacing:", wall_horizontal_spacing.value, "mm")

    print("\n4. HOPPER SLAB")
    print("Main bars:", hopper_main_dia_8.value, "mm")
    print("Main bar spacing:", hopper_main_spacing_8.value, "mm")
    print("Distribution bars:", hopper_distribution_dia_8.value, "mm")
    print("Distribution spacing:",
          hopper_distribution_spacing_8.value, "mm")

    print("\n5. OUTLET SUPPORTING BEAMS")
    print("Number of beams: 4")
    print("Beam size:", beam_b, "mm x", beam_D, "mm")
    print("Main bar dia:", outlet_beam_bar_dia_8.value, "mm")
    print("Stirrup dia:", outlet_beam_stirrup_dia_8.value, "mm")

    print("\n===================================================")


# ------------------------------------------------
# 9. RUN ALL VISUALIZATIONS
# ------------------------------------------------

def run_step8(change=None):

    with step8_output:

        clear_output(wait=True)

        try:
            validate_reinforcement_inputs()

            print("Generating reinforcement drawings...")

            print("\nA. 2D PLAN VIEW")
            draw_plan_view()

            print("\nB. 2D SECTION VIEW")
            draw_section_view()

            print("\nC. 3D REINFORCEMENT MODEL")
            draw_3d_reinforcement()

            print("\nD. REINFORCEMENT SUMMARY")
            show_reinforcement_summary()

        except Exception as e:
            print("ERROR:", str(e))


step8_output = widgets.Output()

step8_button = widgets.Button(
    description="Generate 2D and 3D Reinforcement",
    button_style="success",
    icon="cubes"
)

step8_button.on_click(run_step8)

display(step8_button)
display(step8_output)


Button(button_style='success', description='Generate 2D and 3D Reinforcement', icon='cubes', style=ButtonStyle…

Output()

## 5. Next engineering modules to implement

Before reinforcement design, add and verify these modules in order:

1. **IS 4995 material/load module:** identify the correct material category and obtain bulk density, internal friction, wall friction, pressure ratios and other required parameters from the applicable standard and verified test data.
2. **Load cases:** filling, emptying/discharge, eccentric discharge, non-uniform flow, surcharge, equipment loads, wind and seismic actions where applicable.
3. **Hopper pressure analysis:** resolve pressure normal and tangential to each sloping face, including the governing flow condition.
4. **Structural idealisation:** decide wall support conditions, wall-panel action, hopper plate support, edge beams, outlet framing and column/support arrangement.
5. **RCC design to IS 456:** flexure, shear, axial effects, serviceability, crack control, anchorage, development length, detailing and durability.
6. **Independent verification:** compare key results with hand calculations and a validated structural analysis model.

### References to verify before project use
- IS 4995 (Part 1):1974 – general requirements and assessment of bin loads.
- IS 4995 (Part 2):1974 – design criteria.
- IS 456:2000 – plain and reinforced concrete code of practice, including applicable amendments.
- Relevant loading standards, including applicable parts of IS 875 and IS 1893.

Official BIS preview references:
- [IS 4995 (Part 1) – BIS preview](https://www.services.bis.gov.in/tmp/SR4995_1.pdf)
- [IS 4995 (Part 2) – BIS preview](https://www.services.bis.gov.in/tmp/SR4995_2.pdf)
- [IS 456 – BIS preview](https://www.services.bis.gov.in/tmp/SR456.pdf)

**Engineering disclaimer:** This starter notebook is for learning and preliminary quantity/load exploration. It is not a complete or approved design. Do not construct from its outputs. A qualified structural engineer must establish the design basis, verify the applicable code editions/amendments and independently check all actions, analysis, reinforcement and detailing.